# Phase 2: knowledge-grounded RAG, generation arms

Stage 3 of the Phase 2 plan (`docs/phase2_rag_plan.md`; decision rule: Amendment 5 in `docs/evaluation_protocol.md`).
Same model, greedy decoding and prompt style in every arm; only the disease hint and the retrieved text differ.

| Arm | What the model is given | Questions |
|---|---|---|
| **A1** | the question (style-matched, no hint, no retrieval) | all 130 knowledge questions |
| **A2** | retrieval from the question alone | oracle subset (50) |
| **A3** | the *true* disease named (oracle) | oracle subset |
| **A4** | the *true* disease plus retrieval (oracle) | oracle subset |
| **A5** | the model's *own* disease guess named | all 130 |
| **A6** | the model's own guess plus retrieval: **the system** | all 130 |

A2 to A4 use SLAKE's own disease annotation and are **upper bounds, not results** (Amendment 5). A5 and A6 use only the model's
output.

**Before you run:** run notebook 04 first and paste the `CHOSEN_METHOD` it prints into the build cell below. The retrieval
setting (`topic+textbooks`, 2 passages, 700 characters) was frozen before any model run (plan, section 10).

**Runtime:** free Colab **T4**. About 130 disease guesses (one to 15 passes each) and about 700 short generations. Expect 30 to
60 minutes. Progress is saved to `rag_checkpoint.json` every 20 answers; re-run the cells to resume. `QUICK = True` runs a
16-question smoke test.


## 1. Setup

Identical to the other notebooks. If Colab asks you to restart the runtime after this cell, do so and continue from the next cell.

In [ ]:
#@title Install dependencies (torch is preinstalled on Colab)
!pip install -q "transformers==4.46.3" accelerate einops sentencepiece protobuf pyarrow albumentations huggingface_hub
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> T4 GPU, then re-run this cell.")


## 2. Build the questions and the retriever

The cells below are the repo's own modules, inlined unchanged. Order matters.

In [ ]:
"""Heuristic question-type categorization.

The literature review (CheXagent in particular) found that CXR VLMs score
~90-99% on coarse perception questions (presence/absence, modality, view,
organ identification) but drop to ~77-79% on fine-grained laterality and
severity questions ("which side", "mild vs. severe"). To test whether our
baseline shows the same weak spot, every question in the baseline test set
is tagged with a category up front so results can be broken down by category
rather than reported as one aggregate accuracy number.

This is a deliberately simple keyword heuristic, not a trained classifier.
It will mis-tag some edge cases (e.g. "what is to the left of the heart" is
positional, not a laterality-of-finding question) -- that is an accepted,
documented limitation, not a bug to silently paper over.
"""


import re

LATERALITY_PATTERNS = [
    r"\bleft\b",
    r"\bright\b",
    r"\bbilateral\b",
    r"\bwhich side\b",
    r"\blaterality\b",
    r"\bunilateral\b",
]

SEVERITY_PATTERNS = [
    r"\bmild\b",
    r"\bmoderate\b",
    r"\bsevere\b",
    r"\benlarge(d|ment)?\b",
    r"\bdegree\b",
    r"\bextent\b",
    r"\bhow (much|severe|large|big)\b",
    r"\bsize of\b",
    r"\bworsen(ed|ing)?\b",
    r"\bimprov(ed|ing|ement)\b",
    r"\bsmall\b",
    r"\blarge\b",
]

KNOWLEDGE_PATTERNS = [
    r"\bsymptoms?\b",
    r"\bhow to (treat|prevent|cure)\b",
    r"\b(treatment|therapy|prevention)\b",
    r"\bfunction of\b",
    r"\bcause[sd]?\b",
    r"\bhow do you (determine|diagnose|assess|measure)\b",
    # Added with the knowledge test set: 'why ...?' and 'how can you tell ...?'
    # are answered from anatomy/physiology knowledge. Run 2's saved tags were
    # produced before these two were added.
    r"\bwhy\b",
    r"\bhow can you tell\b",
]

# Questions that are templates about the imaging procedure, not about the
# patient's findings (e.g. SLAKE's "How much damage will be caused to the body
# taking such an image?"). They match the severity keywords ("how much") but are
# not clinical severity, so the data pipeline drops them.
NON_CLINICAL_PATTERNS = [
    r"\bdamage\b.*\btaking such an image\b",
    r"\bradiation\b",
]

CATEGORY_LATERALITY = "laterality"
CATEGORY_SEVERITY = "severity"
CATEGORY_GENERAL = "general"

ALL_CATEGORIES = (CATEGORY_LATERALITY, CATEGORY_SEVERITY, CATEGORY_GENERAL)


def _matches_any(text: str, patterns: list[str]) -> bool:
    return any(re.search(p, text) for p in patterns)


def categorize_question(question: str) -> str:
    """Tag a question as 'laterality', 'severity', or 'general'.

    Laterality is checked before severity because a question can contain
    both kinds of words ("is the left-sided effusion severe?") -- in our
    taxonomy laterality takes priority since it is the harder, more
    specifically-flagged failure mode in CheXagent's own reported numbers.
    """
    q = question.lower().strip()
    if _matches_any(q, LATERALITY_PATTERNS):
        return CATEGORY_LATERALITY
    if _matches_any(q, SEVERITY_PATTERNS):
        return CATEGORY_SEVERITY
    return CATEGORY_GENERAL


def is_knowledge_question(question: str) -> bool:
    """True for questions answered mostly from medical knowledge (symptoms,
    treatment, prevention, how a finding is measured) rather than from looking
    at the image. These are what a retrieval layer can plausibly help with;
    perception questions (is X present, which side) are not.

    Independent of `categorize_question`: a question can be both a laterality
    question and a knowledge question ("How to treat the disease on the lower
    left of lung?")."""
    q = question.lower().strip()
    return _matches_any(q, KNOWLEDGE_PATTERNS)


_SUBTYPE_RULES = [
    ("symptoms", r"\bsymptoms?\b"),
    ("treatment", r"\b(how to treat|treatment|therapy)\b"),
    ("prevention", r"\b(prevent|prevention)\b"),
    ("cause", r"\bcause[sd]?\b"),
    ("function", r"\bfunction of\b"),
    ("explanation", r"\b(why|how can you tell|how do you (determine|diagnose|assess|measure))\b"),
]


def knowledge_subtype(question: str) -> str:
    """Which kind of knowledge question this is (symptoms / treatment /
    prevention / cause / function / explanation / other). Reported separately
    because the SLAKE knowledge questions come from a few fixed templates and
    a model can score well on one template and not another."""
    q = question.lower().strip()
    for name, pattern in _SUBTYPE_RULES:
        if re.search(pattern, q):
            return name
    return "other"


def is_non_clinical_question(question: str) -> bool:
    q = question.lower().strip()
    return _matches_any(q, NON_CLINICAL_PATTERNS)


def answer_format(gold_answer: str) -> str:
    """'yes/no' for closed questions, 'open' otherwise. Reported separately
    because a model that mostly answers 'no' looks decent on yes/no questions
    when most true answers are 'no'."""
    g = gold_answer.lower().strip().rstrip(".")
    return "yes/no" if g in ("yes", "no") else "open"


In [ ]:
"""Build a small, chest-X-ray-only VQA test set for Phase 1 baseline testing.

Important caveat (documented here, not hidden): CheXagent's own training
corpus (CheXinstruct) explicitly rolls in VQA-RAD, SLAKE, and Open-I/IU X-ray
among its 32 source datasets. So evaluating CheXagent on these test splits is
*not* a clean held-out/zero-shot measurement -- it is close to in-distribution.
That is simply the state of public CXR VQA benchmarks today (virtually every
other paper in our review reuses the same splits for the same reason).

The point of Phase 1 is not to claim a novel generalization number; it is to
check whether our own baseline run reproduces the *specific, already-known*
weak spot from CheXagent's paper (strong on coarse/general questions, weaker
on laterality/severity) on data we control end-to-end, so Phase 2 (RAG) has a
concrete, measured gap to target rather than a hypothetical one.

Two sources are combined:
- SLAKE (test + validation): has structured `location` + `modality` metadata, so
  chest X-ray rows can be filtered precisely (no guessing from question text).
- VQA-RAD: has no organ metadata, so chest rows are selected with a keyword
  heuristic (noisier, clearly flagged below -- not a silent assumption).

Non-clinical template questions (e.g. SLAKE's radiation-damage question) are
dropped, repeated question text is capped per category, and every record gets a
stable `id` taken from the source dataset's own row id so later runs (RAG) can
be compared with this baseline row by row.
"""


import hashlib
import json
import random
import re
import time
import zipfile
from pathlib import Path

import requests


SLAKE_DATASET = "BoKelvin/SLAKE"
SLAKE_IMAGES_ZIP_URL = "https://huggingface.co/datasets/BoKelvin/SLAKE/resolve/main/imgs.zip"
VQA_RAD_DATASET = "flaviagiammarino/vqa-rad"
DATASETS_SERVER_ROWS_URL = "https://datasets-server.huggingface.co/rows"

# Preferred data path: download each dataset's own files once (5 requests).
# The paging API (`datasets-server`, ~150 requests) rate-limits Colab with HTTP
# 429, so it is kept only as a fallback. Row ids are identical on both paths,
# checked against all 224 question ids of run 2.
HF_RESOLVE_URL = "https://huggingface.co/datasets/{repo}/resolve/main/{path}"
SLAKE_SPLIT_FILES = {"test": "test.json", "validation": "validation.json", "train": "train.json"}
VQA_RAD_PARQUET_FILES = {
    "test": "data/test-00000-of-00001-e5bc3d208bb4deeb.parquet",
    "train": "data/train-00000-of-00001-eb8844602202be60.parquet",
}
VQA_RAD_PARQUET_KEY_PREFIX = "vqarad-parquet"
RAW_CACHE_DIR = Path("raw_cache")

# SLAKE's `location` field uses these labels for chest X-ray images.
SLAKE_CHEST_LOCATIONS = {"Chest_heart", "Chest_lung", "Chest_mediastinal", "Lung"}

# VQA-RAD keyword heuristic: a question must mention a chest-related term and
# must NOT mention a clearly non-chest body region. This is approximate --
# some true negatives/positives will slip through -- which is why SLAKE
# (precise metadata) is the primary source and VQA-RAD is secondary/noisier.
VQA_RAD_CHEST_KEYWORDS = {
    "lung", "lungs", "chest", "pulmonary", "pleural", "pneumothorax",
    "cardiac", "heart", "mediastin", "rib", "costophrenic", "consolidation",
    "effusion", "atelectasis", "diaphragm", "thora", "cardiomegaly",
    "pneumonia", "hilar", "bronch",
}
# Keywords that must match as a whole word. "rib" is the one that matters: as a
# substring it matched "describe", "circumscribed", "distribution" and
# "tributaries", which let brain/abdominal "describe the lesion" questions into
# the chest set (found by looking at the images of the side set).
VQA_RAD_WHOLE_WORD_KEYWORDS = {"rib", "ribs"}
_WHOLE_WORD_RE = re.compile(r"\b(" + "|".join(sorted(VQA_RAD_WHOLE_WORD_KEYWORDS)) + r")\b")

VQA_RAD_EXCLUDE_KEYWORDS = {
    "brain", "skull", "head", "abdomen", "abdominal", "liver", "kidney",
    "pelvis", "pelvic", "spine", "spinal", "bowel", "intestine", "stomach",
    "bladder", "uterus", "ovary", "appendix", "cerebral", "cranial", "sinus",
    # Added after the pilot run surfaced non-chest items (e.g. "vascular
    # distribution of these infarcts"). Still a heuristic -- review the images.
    "infarct", "ventric", "sulci", "gyri", "white matter", "gray matter",
    "grey matter", "falx", "midline", "mass effect", "vascular distribution",
    "hepatic", "splenic", "spleen", "adrenal", "pancrea", "gallbladder",
    "renal", "psoas", "femur", "knee", "shoulder", "orbit",
}


def mentions_chest_term(question: str) -> bool:
    """True if the question text contains a chest keyword (substring match,
    except for short words listed in VQA_RAD_WHOLE_WORD_KEYWORDS)."""
    q = question.lower()
    if _WHOLE_WORD_RE.search(q):
        return True
    return any(k in q for k in VQA_RAD_CHEST_KEYWORDS if k not in VQA_RAD_WHOLE_WORD_KEYWORDS)


def _retry_wait(resp, attempt: int, backoff: float) -> float:
    """Seconds to wait before retrying. Honours Retry-After on HTTP 429 and
    otherwise backs off exponentially (capped at 60 s)."""
    if resp is not None and resp.status_code == 429:
        try:
            return min(120.0, max(float(resp.headers.get("Retry-After", 0)), backoff * 4 * (attempt + 1)))
        except ValueError:
            pass
    return min(60.0, backoff * (2 ** attempt))


def _download_file(url: str, dest: str | Path, retries: int = 8, backoff: float = 2.0) -> Path:
    """Download `url` to `dest` once (skipped if present), streaming, with
    retries on 429 / 5xx / connection errors. Written via a temp file so an
    interrupted download is never mistaken for a complete one."""
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    last_exc: Exception | None = None
    for attempt in range(retries):
        resp = None
        try:
            resp = requests.get(url, stream=True, timeout=180)
            if resp.status_code == 200:
                tmp = dest.with_suffix(dest.suffix + ".part")
                with open(tmp, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=1 << 20):
                        f.write(chunk)
                tmp.replace(dest)
                return dest
            if resp.status_code not in (429, 500, 502, 503, 504):
                resp.raise_for_status()
        except requests.RequestException as exc:
            last_exc = exc
            if resp is not None and resp.status_code not in (429, 500, 502, 503, 504):
                raise
        time.sleep(_retry_wait(resp, attempt, backoff))
    raise last_exc or RuntimeError(f"could not download {url}")


def _get_rows(dataset: str, split: str, offset: int, length: int = 100,
              retries: int = 8, backoff: float = 1.5) -> dict:
    last_exc = None
    for attempt in range(retries):
        try:
            resp = requests.get(
                DATASETS_SERVER_ROWS_URL,
                params={
                    "dataset": dataset, "config": "default", "split": split,
                    "offset": offset, "length": length,
                },
                timeout=30,
            )
            if resp.status_code == 200:
                return resp.json()
        except requests.RequestException as exc:
            last_exc = exc
            resp = None
        time.sleep(_retry_wait(resp, attempt, backoff))
    if last_exc:
        raise last_exc
    resp.raise_for_status()
    return {}


def _iter_all_rows(dataset: str, split: str, page_size: int = 100):
    offset = 0
    while True:
        data = _get_rows(dataset, split, offset, length=page_size)
        rows = data.get("rows", [])
        if not rows:
            return
        for row in rows:
            yield {**row["row"], "_row_idx": row["row_idx"]}
        offset += len(rows)
        if offset >= data.get("num_rows_total", 0):
            return


def _iter_slake_rows(split: str):
    """SLAKE rows from the dataset's own JSON file; paging API as fallback."""
    try:
        url = HF_RESOLVE_URL.format(repo=SLAKE_DATASET, path=SLAKE_SPLIT_FILES[split])
        path = _download_file(url, RAW_CACHE_DIR / f"slake_{split}.json")
        rows = json.loads(path.read_text())
    except Exception as exc:
        print(f"[data] SLAKE {split}: file download failed ({exc!r}); falling back to the paging API")
        yield from _iter_all_rows(SLAKE_DATASET, split)
        return
    yield from rows


def fetch_slake_chest_records(split: str = "test") -> list[dict]:
    """English chest-X-ray rows from SLAKE, filtered via structured metadata."""
    records = []
    for rr in _iter_slake_rows(split):
        if (
            rr.get("modality") == "X-Ray"
            and rr.get("location") in SLAKE_CHEST_LOCATIONS
            and rr.get("q_lang") == "en"
            and not is_non_clinical_question(rr["question"])
        ):
            records.append({
                "source_dataset": "SLAKE",
                "source_split": split,
                "question": rr["question"],
                "gold_answer": rr["answer"],
                "answer_type": rr.get("answer_type"),
                "uid": f"SLAKE_{rr['qid']}",
                "image_key": rr["img_name"],  # e.g. "xmlab102/source.jpg"
                "category": categorize_question(rr["question"]),
                "answer_format": answer_format(rr["answer"]),
                "is_knowledge": is_knowledge_question(rr["question"]),
            })
    return records


def _vqa_rad_parquet_path(split: str) -> Path:
    url = HF_RESOLVE_URL.format(repo=VQA_RAD_DATASET, path=VQA_RAD_PARQUET_FILES[split])
    return _download_file(url, RAW_CACHE_DIR / f"vqa_rad_{split}.parquet")


def _iter_vqa_rad_rows(split: str):
    """VQA-RAD rows from the dataset's parquet file (images are embedded, so
    there are no expiring signed URLs, and an image's content hash gives a real
    image identity); paging API as fallback if pyarrow or the download fails.

    Yields dicts with question, answer, _row_idx, _image_key, _image_id."""
    try:
        import pyarrow.parquet as pq
        table = pq.read_table(_vqa_rad_parquet_path(split))
        images, questions, answers = table.column("image"), table.column("question"), table.column("answer")
        rows = []
        for idx in range(table.num_rows):
            img_bytes = images[idx].as_py()["bytes"]
            rows.append({
                "question": questions[idx].as_py(), "answer": answers[idx].as_py(), "_row_idx": idx,
                "_image_key": f"{VQA_RAD_PARQUET_KEY_PREFIX}:{split}:{idx}",
                "_image_id": hashlib.md5(img_bytes).hexdigest()[:12],
            })
    except Exception as exc:
        print(f"[data] VQA-RAD {split}: parquet path failed ({exc!r}); falling back to the paging API")
        for rr in _iter_all_rows(VQA_RAD_DATASET, split):
            yield {"question": rr["question"], "answer": rr["answer"], "_row_idx": rr["_row_idx"],
                   "_image_key": rr["image"]["src"], "_image_id": None}
        return
    yield from rows


def fetch_vqa_rad_chest_records(splits: tuple[str, ...] = ("test", "train")) -> list[dict]:
    """Chest-ish rows from VQA-RAD, filtered via keyword heuristic (noisier)."""
    records = []
    for split in splits:
        for rr in _iter_vqa_rad_rows(split):
            q = rr["question"].lower()
            if (
                mentions_chest_term(q)
                and not any(k in q for k in VQA_RAD_EXCLUDE_KEYWORDS)
                and not is_non_clinical_question(rr["question"])
            ):
                records.append({
                    "source_dataset": "VQA-RAD",
                    "source_split": split,
                    "question": rr["question"],
                    "gold_answer": rr["answer"],
                    "answer_type": None,
                    "uid": f"VQA-RAD_{split}_{rr['_row_idx']}",
                    "image_key": rr["_image_key"],
                    "image_id": rr["_image_id"],  # content hash of the image (None on the fallback path)
                    "category": categorize_question(rr["question"]),
                    "answer_format": answer_format(rr["answer"]),
                    "is_knowledge": is_knowledge_question(rr["question"]),
                })
    return records


def stratified_sample(records: list[dict], n_per_category: int, seed: int = 0,
                      max_per_question: int = 4) -> list[dict]:
    """Sample up to `n_per_category` records per category.

    `max_per_question` caps how many rows with identical question text can
    be drawn (the pilot run had "is the heart enlarged?" ~12 times in 60 rows,
    which lets one question dominate a category). Deterministic for a given
    input order and seed.
    """
    rng = random.Random(seed)
    by_cat: dict[str, list[dict]] = {}
    for r in records:
        by_cat.setdefault(r["category"], []).append(r)
    sampled = []
    for cat in sorted(by_cat):
        shuffled = by_cat[cat][:]
        rng.shuffle(shuffled)
        per_question: dict[str, int] = {}
        taken = []
        for r in shuffled:
            key = r["question"].lower().strip()
            if per_question.get(key, 0) >= max_per_question:
                continue
            per_question[key] = per_question.get(key, 0) + 1
            taken.append(r)
            if len(taken) >= n_per_category:
                break
        sampled.extend(taken)
    rng.shuffle(sampled)
    return sampled


def pool_summary(records: list[dict]) -> dict[str, dict[str, int]]:
    """Counts per category / source / answer format, so it is visible how many
    questions of each kind actually exist (severity questions are rare in
    public CXR VQA sets -- that is itself a finding)."""
    summary: dict[str, dict[str, int]] = {"category": {}, "source": {}, "answer_format": {}}
    for r in records:
        for field, key in (("category", r["category"]), ("source", r["source_dataset"]),
                           ("answer_format", r["answer_format"])):
            summary[field][key] = summary[field].get(key, 0) + 1
    return summary


def download_slake_images(records: list[dict], dest_dir: str | Path,
                           zip_cache_path: str | Path | None = None) -> dict[str, str]:
    """Download SLAKE's imgs.zip once (cached) and extract only needed files."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    needed_keys = {r["image_key"] for r in records if r["source_dataset"] == "SLAKE"}
    if not needed_keys:
        return {}

    zip_cache_path = Path(zip_cache_path) if zip_cache_path else dest_dir / "_slake_imgs.zip"
    _download_file(SLAKE_IMAGES_ZIP_URL, zip_cache_path)

    local_paths = {}
    with zipfile.ZipFile(zip_cache_path) as zf:
        for key in needed_keys:
            member = f"imgs/{key}"
            out_path = dest_dir / key.replace("/", "__")
            if not out_path.exists():
                with zf.open(member) as src, open(out_path, "wb") as dst:
                    dst.write(src.read())
            local_paths[key] = str(out_path)
    return local_paths


def download_vqa_rad_images(records: list[dict], dest_dir: str | Path) -> dict[str, str]:
    """Write each needed VQA-RAD image to `dest_dir`. Records from the parquet
    path are extracted from the parquet file and named by image content hash, so
    questions about the same image share one file. Records from the paging
    fallback are fetched from their (expiring) signed URLs."""
    dest_dir = Path(dest_dir)
    dest_dir.mkdir(parents=True, exist_ok=True)
    local_paths: dict[str, str] = {}
    tables: dict[str, object] = {}
    for r in records:
        if r["source_dataset"] != "VQA-RAD":
            continue
        key = r["image_key"]
        if key in local_paths:
            continue
        if key.startswith(VQA_RAD_PARQUET_KEY_PREFIX + ":"):
            _, split, idx = key.split(":")
            out_path = dest_dir / f"vqarad_{r['image_id']}.jpg"
            if not out_path.exists():
                if split not in tables:
                    import pyarrow.parquet as pq
                    tables[split] = pq.read_table(_vqa_rad_parquet_path(split), columns=["image"])
                out_path.write_bytes(tables[split].column("image")[int(idx)].as_py()["bytes"])
        else:
            digest = hashlib.md5(key.encode()).hexdigest()[:12]
            out_path = dest_dir / f"vqarad_{digest}.jpg"
            if not out_path.exists():
                resp = requests.get(key, timeout=60)
                resp.raise_for_status()
                with open(out_path, "wb") as f:
                    f.write(resp.content)
        local_paths[key] = str(out_path)
    return local_paths


def build_baseline_test_set(images_dir: str | Path, n_per_category: int = 100,
                             seed: int = 0, max_per_question: int = 4,
                             verbose: bool = True) -> list[dict]:
    """End-to-end: fetch metadata, stratify-sample per category, download
    images, return records ready for model inference. Each record has an
    `image_path` and a stable `id` (derived from the source dataset's own row
    id, so baseline and later RAG runs can be compared row by row)."""
    slake = (fetch_slake_chest_records(split="test")
             + fetch_slake_chest_records(split="validation"))
    vqa_rad = fetch_vqa_rad_chest_records(splits=("test", "train"))
    all_records = slake + vqa_rad
    if verbose:
        print("Available pool before sampling:", pool_summary(all_records))

    sampled = stratified_sample(all_records, n_per_category=n_per_category, seed=seed,
                                max_per_question=max_per_question)

    slake_records = [r for r in sampled if r["source_dataset"] == "SLAKE"]
    vqa_rad_records = [r for r in sampled if r["source_dataset"] == "VQA-RAD"]
    slake_paths = download_slake_images(slake_records, images_dir)
    vqa_rad_paths = download_vqa_rad_images(vqa_rad_records, images_dir)

    for r in sampled:
        paths = slake_paths if r["source_dataset"] == "SLAKE" else vqa_rad_paths
        r["image_path"] = paths[r["image_key"]]
        r["id"] = r["uid"]

    return sampled


In [ ]:
"""Answer scoring for the baseline run.

Med-Flamingo's own finding (clinician ratings disagreeing sharply with
automated text-overlap metrics) is why several automated scores are reported
side by side rather than one. None is a substitute for a human spot-check.

- `exact_match`: strict, normalized string equality. Fair for yes/no, harsh for
  open-ended answers.
- `loose_match`: substring containment either direction after normalization.
- `token_f1`: word-overlap F1 (SQuAD style). Gives partial credit to open
  answers like "Chest tightness, shortness of breath" vs "Chest tightness,
  fatigue", which exact_match scores 0.

Every accuracy is reported with a Wilson 95% confidence interval, because with
a few dozen questions per bucket most differences between buckets are not
statistically meaningful, and the report should say so.

Yes/no questions get extra treatment (sensitivity/specificity and the
"always answer the majority class" baseline): a model that answers "no" to
almost everything scores well when most true answers are "no", but misses
real findings.
"""


import math
import re
from dataclasses import dataclass, field

_YES_SYNONYMS = {"yes", "y", "yeah", "affirmative", "true", "present"}
_NO_SYNONYMS = {"no", "n", "nope", "negative", "false", "absent", "not present"}

_PUNCT_RE = re.compile(r"[^a-z0-9\s]")
_WS_RE = re.compile(r"\s+")


def normalize(text: str) -> str:
    text = text.lower().strip()
    text = _PUNCT_RE.sub(" ", text)
    text = _WS_RE.sub(" ", text).strip()
    if text in _YES_SYNONYMS:
        return "yes"
    if text in _NO_SYNONYMS:
        return "no"
    return text


def exact_match(prediction: str, gold: str) -> bool:
    return normalize(prediction) == normalize(gold)


def loose_match(prediction: str, gold: str) -> bool:
    p, g = normalize(prediction), normalize(gold)
    if not p or not g:
        return False
    if p == g:
        return True
    # Guard against trivially short strings matching almost anything.
    shorter, longer = (p, g) if len(p) <= len(g) else (g, p)
    if len(shorter) < 3:
        return False
    return shorter in longer


def token_f1(prediction: str, gold: str) -> float:
    p_tokens, g_tokens = normalize(prediction).split(), normalize(gold).split()
    if not p_tokens or not g_tokens:
        return float(p_tokens == g_tokens)
    remaining = list(g_tokens)
    common = 0
    for tok in p_tokens:
        if tok in remaining:
            remaining.remove(tok)
            common += 1
    if common == 0:
        return 0.0
    precision, recall = common / len(p_tokens), common / len(g_tokens)
    return 2 * precision * recall / (precision + recall)


def wilson_interval(correct: int, total: int, z: float = 1.96) -> tuple[float, float]:
    if total == 0:
        return (float("nan"), float("nan"))
    p = correct / total
    denom = 1 + z * z / total
    center = (p + z * z / (2 * total)) / denom
    half = z * math.sqrt(p * (1 - p) / total + z * z / (4 * total * total)) / denom
    return (max(0.0, center - half), min(1.0, center + half))


@dataclass
class Stats:
    total: int = 0
    exact_correct: int = 0
    loose_correct: int = 0
    f1_sum: float = 0.0

    def add(self, em: bool, lm: bool, f1: float) -> None:
        self.total += 1
        self.exact_correct += int(em)
        self.loose_correct += int(lm)
        self.f1_sum += f1

    @property
    def exact_accuracy(self) -> float:
        return self.exact_correct / self.total if self.total else float("nan")

    @property
    def loose_accuracy(self) -> float:
        return self.loose_correct / self.total if self.total else float("nan")

    @property
    def mean_f1(self) -> float:
        return self.f1_sum / self.total if self.total else float("nan")

    @property
    def exact_ci(self) -> tuple[float, float]:
        return wilson_interval(self.exact_correct, self.total)


# Backwards-compatible name used by earlier notebook versions and tests.
CategoryStats = Stats


@dataclass
class YesNoStats:
    """Confusion counts for yes/no questions, treating 'yes' as the positive
    class (i.e. 'there is a finding')."""
    tp: int = 0   # gold yes, predicted yes
    fn: int = 0   # gold yes, predicted not-yes  (missed finding)
    tn: int = 0   # gold no,  predicted no
    fp: int = 0   # gold no,  predicted not-no   (false alarm)

    @property
    def total(self) -> int:
        return self.tp + self.fn + self.tn + self.fp

    @property
    def sensitivity(self) -> float:
        pos = self.tp + self.fn
        return self.tp / pos if pos else float("nan")

    @property
    def specificity(self) -> float:
        neg = self.tn + self.fp
        return self.tn / neg if neg else float("nan")

    @property
    def accuracy(self) -> float:
        return (self.tp + self.tn) / self.total if self.total else float("nan")

    @property
    def majority_baseline(self) -> float:
        """Accuracy of always answering the more common gold class."""
        return max(self.tp + self.fn, self.tn + self.fp) / self.total if self.total else float("nan")

    @property
    def predicted_yes_rate(self) -> float:
        return (self.tp + self.fp) / self.total if self.total else float("nan")

    @property
    def gold_yes_rate(self) -> float:
        return (self.tp + self.fn) / self.total if self.total else float("nan")


@dataclass
class EvaluationReport:
    per_category: dict[str, Stats] = field(default_factory=dict)
    per_source: dict[str, Stats] = field(default_factory=dict)
    per_format: dict[str, Stats] = field(default_factory=dict)
    per_category_format: dict[str, Stats] = field(default_factory=dict)
    per_kind: dict[str, Stats] = field(default_factory=dict)
    overall: Stats = field(default_factory=Stats)
    yes_no: YesNoStats = field(default_factory=YesNoStats)
    yes_no_by_category: dict[str, YesNoStats] = field(default_factory=dict)
    rows: list[dict] = field(default_factory=list)


def _gold_format(rec: dict) -> str:
    if rec.get("answer_format"):
        return rec["answer_format"]
    return "yes/no" if normalize(rec["gold_answer"]) in ("yes", "no") else "open"


def evaluate(records: list[dict]) -> EvaluationReport:
    """Score a list of records.

    Required keys: 'question', 'gold_answer', 'prediction', 'category',
    'source_dataset'. Optional: 'answer_format' ('yes/no'|'open', inferred if
    absent) and 'is_knowledge' (bool). Returns overall and per-bucket stats plus
    per-row scoring so a human can inspect individual disagreements.
    """
    report = EvaluationReport()

    for rec in records:
        em = exact_match(rec["prediction"], rec["gold_answer"])
        lm = loose_match(rec["prediction"], rec["gold_answer"])
        f1 = token_f1(rec["prediction"], rec["gold_answer"])
        fmt = _gold_format(rec)
        kind = "knowledge" if rec.get("is_knowledge") else "perception"

        for bucket, key in (
            (report.per_category, rec["category"]),
            (report.per_source, rec["source_dataset"]),
            (report.per_format, fmt),
            (report.per_category_format, f"{rec['category']} / {fmt}"),
            (report.per_kind, kind),
        ):
            bucket.setdefault(key, Stats()).add(em, lm, f1)
        report.overall.add(em, lm, f1)

        if fmt == "yes/no":
            gold_yes = normalize(rec["gold_answer"]) == "yes"
            pred = normalize(rec["prediction"])
            for yn in (report.yes_no, report.yes_no_by_category.setdefault(rec["category"], YesNoStats())):
                if gold_yes:
                    yn.tp += int(pred == "yes")
                    yn.fn += int(pred != "yes")
                else:
                    yn.tn += int(pred == "no")
                    yn.fp += int(pred != "no")

        report.rows.append({**rec, "answer_format": fmt, "exact_match": em,
                            "loose_match": lm, "token_f1": round(f1, 3)})

    return report


def _fmt_stats(s: Stats) -> str:
    lo, hi = s.exact_ci
    return (f"n={s.total:<4} exact={s.exact_accuracy:6.1%} (95% CI {lo:.0%}-{hi:.0%})"
            f"  loose={s.loose_accuracy:6.1%}  token_f1={s.mean_f1:.2f}")


def format_report(report: EvaluationReport) -> str:
    L = []
    L.append("=== Overall ===")
    L.append(_fmt_stats(report.overall))

    def section(title, bucket):
        L.append("")
        L.append(f"=== {title} ===")
        for key, s in sorted(bucket.items()):
            L.append(f"{key:>24}: {_fmt_stats(s)}")

    section("By question category (laterality / severity / general)", report.per_category)
    section("By answer format (yes/no vs open-ended)", report.per_format)
    section("By category and answer format", report.per_category_format)
    section("By question kind (knowledge = symptoms/treatment/etc; perception = look at the image)", report.per_kind)
    section("By source dataset", report.per_source)

    yn = report.yes_no
    L.append("")
    L.append("=== Yes/no questions: does the model miss findings? ===")
    if yn.total:
        L.append(f"n={yn.total}  accuracy={yn.accuracy:.1%}  "
                 f"always-answer-majority baseline={yn.majority_baseline:.1%}")
        L.append(f"sensitivity (correct on gold 'yes')={yn.sensitivity:.1%}  "
                 f"specificity (correct on gold 'no')={yn.specificity:.1%}")
        L.append(f"gold 'yes' rate={yn.gold_yes_rate:.1%}  model predicted 'yes' rate={yn.predicted_yes_rate:.1%}")
        for cat, c in sorted(report.yes_no_by_category.items()):
            L.append(f"  {cat:>12}: n={c.total:<3} acc={c.accuracy:.0%}  sens={c.sensitivity:.0%}  spec={c.specificity:.0%}")
    else:
        L.append("(no yes/no questions in this run)")
    L.append("")
    L.append("Note: buckets with n < ~50 have wide confidence intervals; treat differences that "
             "the intervals overlap on as unproven.")
    return "\n".join(L)


In [ ]:
"""Small, dependency-free statistical tests used to compare buckets and runs.

Everything here is implemented with the standard library so it runs in the CI
job and in Colab without scipy. Each function is checked against known values
in `tests/test_stats.py`.
"""


import math
import random


def binom_pmf(k: int, n: int, p: float) -> float:
    if p <= 0.0:
        return float(k == 0)
    if p >= 1.0:
        return float(k == n)
    return math.comb(n, k) * (p ** k) * ((1 - p) ** (n - k))


def binom_two_sided(k: int, n: int, p: float = 0.5) -> float:
    """Exact two-sided binomial test: probability, under Binomial(n, p), of an
    outcome no more likely than the observed `k`."""
    if n == 0:
        return 1.0
    observed = binom_pmf(k, n, p)
    total = sum(binom_pmf(i, n, p) for i in range(n + 1) if binom_pmf(i, n, p) <= observed * (1 + 1e-9))
    return min(1.0, total)


def mcnemar_exact(only_a_correct: int, only_b_correct: int) -> float:
    """Exact McNemar test on discordant pairs: under 'no difference between the
    two systems', each discordant pair is equally likely to favour either."""
    return binom_two_sided(only_a_correct, only_a_correct + only_b_correct, 0.5)


def fisher_exact_2x2(a: int, b: int, c: int, d: int) -> float:
    """Two-sided Fisher exact test for the table [[a, b], [c, d]]."""
    n1, n2, k = a + b, c + d, a + c
    n = n1 + n2

    def pmf(x: int) -> float:
        return math.comb(n1, x) * math.comb(n2, k - x) / math.comb(n, k)

    lo, hi = max(0, k - n2), min(n1, k)
    observed = pmf(a)
    return min(1.0, sum(pmf(x) for x in range(lo, hi + 1) if pmf(x) <= observed * (1 + 1e-9)))


def cluster_bootstrap_mean_diff(diffs: list[float], clusters: list[str], n_boot: int = 10000,
                                seed: int = 0, alpha: float = 0.05) -> tuple[float, float, float]:
    """Mean of paired differences with a percentile bootstrap CI that resamples
    whole clusters (e.g. all questions about one image), because questions about
    the same image or from the same template are not independent. Returns
    (mean, ci_low, ci_high)."""
    if not diffs:
        return (float("nan"), float("nan"), float("nan"))
    groups: dict[str, list[float]] = {}
    for d, c in zip(diffs, clusters):
        groups.setdefault(c, []).append(d)
    keys = list(groups)
    rng = random.Random(seed)
    means = []
    for _ in range(n_boot):
        total, count = 0.0, 0
        for _ in keys:
            g = groups[keys[rng.randrange(len(keys))]]
            total += sum(g)
            count += len(g)
        means.append(total / count)
    means.sort()
    lo = means[int((alpha / 2) * n_boot)]
    hi = means[min(n_boot - 1, int((1 - alpha / 2) * n_boot))]
    return (sum(diffs) / len(diffs), lo, hi)


In [ ]:
"""Scoring for 'which side' questions.

Gold answers to side questions range from "Right" to "Lower Left Lung", and a
model that answers "left lung" to "Lower Left Lung" has the side right even
though exact match scores it 0. So side questions are scored on the extracted
side (left / right / both), in addition to the usual metrics.

The same extraction supports the specific follow-up question from run 2: when
the correct side is left or both, does the model still answer "right"?
"""


import re


_BOTH_RE = re.compile(r"\b(bilateral|bilaterally|both)\b")
_LEFT_RE = re.compile(r"\bleft\b")
_RIGHT_RE = re.compile(r"\bright\b")


def side_of(text: str) -> str | None:
    """'left', 'right' or 'both' if the text names a side, else None.

    Text naming both words ("left and right lungs") counts as 'both'. Answers
    such as "No" or "Lung" name no side and return None."""
    t = str(text).lower()
    if _BOTH_RE.search(t):
        return "both"
    has_l, has_r = bool(_LEFT_RE.search(t)), bool(_RIGHT_RE.search(t))
    if has_l and has_r:
        return "both"
    if has_l:
        return "left"
    if has_r:
        return "right"
    return None


def score_side_rows(rows: list[dict]) -> list[dict]:
    """Add gold_side / pred_side / side_correct / side_given to each row."""
    out = []
    for r in rows:
        gold, pred = side_of(r["gold_answer"]), side_of(r["prediction"])
        out.append({**r, "gold_side": gold, "pred_side": pred,
                    "side_given": pred is not None, "side_correct": pred == gold})
    return out


def side_report(rows: list[dict]) -> dict:
    """Confusion table and bias statistics for rows scored by `score_side_rows`
    (rows whose gold names no side are ignored)."""
    rows = [r for r in rows if r.get("gold_side")]
    sides = ("left", "right", "both")
    confusion = {g: {p: 0 for p in (*sides, "none")} for g in sides}
    for r in rows:
        confusion[r["gold_side"]][r["pred_side"] or "none"] += 1

    per_gold = {}
    for g in sides:
        n = sum(confusion[g].values())
        per_gold[g] = {"n": n, "correct": confusion[g][g],
                       "accuracy": confusion[g][g] / n if n else float("nan")}

    # Asymmetry: among gold-left items that got a left/right answer, how often
    # was it "right"; among gold-right items, how often "left"? Equal rates
    # mean errors are symmetric; a higher left->right rate is a right default.
    l_to_r, l_to_l = confusion["left"]["right"], confusion["left"]["left"]
    r_to_l, r_to_r = confusion["right"]["left"], confusion["right"]["right"]
    asym_p = (fisher_exact_2x2(l_to_r, l_to_l, r_to_l, r_to_r)
              if (l_to_r + l_to_l) and (r_to_l + r_to_r) else float("nan"))

    # Among left/right-gold items answered left or right, is accuracy above 50%?
    lr = [r for r in rows if r["gold_side"] in ("left", "right") and r["pred_side"] in ("left", "right")]
    lr_correct = sum(r["side_correct"] for r in lr)

    pred_right = sum(1 for r in rows if r["pred_side"] == "right")
    gold_right = sum(1 for r in rows if r["gold_side"] == "right")
    return {
        "n": len(rows),
        "confusion": confusion,
        "per_gold": per_gold,
        "left_to_right_rate": l_to_r / (l_to_r + l_to_l) if (l_to_r + l_to_l) else float("nan"),
        "right_to_left_rate": r_to_l / (r_to_l + r_to_r) if (r_to_l + r_to_r) else float("nan"),
        "asymmetry_p": asym_p,
        "lr_answered": len(lr),
        "lr_correct": lr_correct,
        "lr_vs_chance_p": binom_two_sided(lr_correct, len(lr), 0.5) if lr else float("nan"),
        "predicted_right_rate": pred_right / len(rows) if rows else float("nan"),
        "gold_right_rate": gold_right / len(rows) if rows else float("nan"),
        "no_side_answer": sum(1 for r in rows if r["pred_side"] is None),
    }


def format_side_report(rep: dict) -> str:
    L = [f"=== Side questions (n={rep['n']}) ===", "gold \\ predicted      left  right  both  none  accuracy"]
    for g in ("left", "right", "both"):
        c, pg = rep["confusion"][g], rep["per_gold"][g]
        L.append(f"{g:>20}  {c['left']:5d}  {c['right']:5d}  {c['both']:4d}  {c['none']:4d}"
                 f"  {pg['correct']}/{pg['n']} ({pg['accuracy']:.0%})" if pg["n"] else f"{g:>20}  (none)")
    L.append("")
    L.append(f"model answered 'right' on {rep['predicted_right_rate']:.0%} of questions; "
             f"'right' was correct on {rep['gold_right_rate']:.0%}")
    L.append(f"gold-left answered 'right': {rep['left_to_right_rate']:.0%}   "
             f"gold-right answered 'left': {rep['right_to_left_rate']:.0%}   "
             f"(Fisher p for asymmetry = {rep['asymmetry_p']:.3f})")
    L.append(f"left-vs-right accuracy among items answered left/right: "
             f"{rep['lr_correct']}/{rep['lr_answered']}   (exact binomial p vs 50% chance = {rep['lr_vs_chance_p']:.3f})")
    L.append(f"questions where the model named no side: {rep['no_side_answer']}")
    return "\n".join(L)


In [ ]:
"""Two supplementary test sets built from the same public sources as the 224-question baseline.

1. Knowledge set: every symptoms / treatment / prevention / cause / function /
   'why' question available in SLAKE (all splits) and VQA-RAD. Run 2 had only 24,
   too few to detect a retrieval improvement. The pool is small (about 130) and
   made of a few fixed templates, so analysis must treat questions about the
   same image or template as correlated (see compare.py).

2. Side set: open questions whose gold answer names a side ('Right', 'Lower Left
   Lung', 'Both Lung'), sampled with equal numbers of left and right answers so
   a model that favours one side cannot score well by habit. Run 2 hinted at a
   right-side default from only 13 such questions.

Both sets include SLAKE train rows. CheXagent was very likely trained on those
(CheXinstruct includes SLAKE), so every record carries `source_split` and
results should be reported for train and test/validation rows separately.
Train rows make the set larger, not cleaner.
"""


import hashlib
import json
import random
from pathlib import Path


KNOWLEDGE_PROMPT = "{question}"
SIDE_PROMPT = "{question} Answer concisely based only on what is visible in the image."


def prompt_for_set(set_name: str, question: str, category: str) -> str:
    """Prompt used when running a supplementary set.

    knowledge: the plain question for every row. Run 2 appended "based only on
    what is visible in the image" to laterality/severity-tagged rows, which
    included 20 of its 24 knowledge questions, a confound for a question that
    is answered from knowledge.
    side: the visible-in-image suffix for every row (what run 2 used for
    laterality questions).
    """
    if set_name == "knowledge":
        return KNOWLEDGE_PROMPT.format(question=question)
    if set_name == "side":
        return SIDE_PROMPT.format(question=question)
    raise ValueError(f"unknown set: {set_name}")


def fetch_full_pool(include_train: bool = True, cache_path: str | Path | None = None) -> list[dict]:
    """All chest candidates from SLAKE (test, validation, train) and VQA-RAD.
    Cached as JSON because the full fetch takes about a minute."""
    if cache_path and Path(cache_path).exists():
        return json.loads(Path(cache_path).read_text())
    splits = ["test", "validation"] + (["train"] if include_train else [])
    pool: list[dict] = []
    for split in splits:
        pool += fetch_slake_chest_records(split=split)
    pool += fetch_vqa_rad_chest_records(splits=("test", "train"))
    if cache_path:
        Path(cache_path).parent.mkdir(parents=True, exist_ok=True)
        Path(cache_path).write_text(json.dumps(pool))
    return pool


def _image_identity(r: dict) -> str:
    """Stable identity of the image behind a question. VQA-RAD rows carry a
    content hash, so questions about one image are recognised as sharing it."""
    return f"{r['source_dataset']}:{r.get('image_id') or r['image_key'].split('?')[0]}"


def select_knowledge_records(pool: list[dict]) -> list[dict]:
    """Knowledge questions, deduplicated on (source, image, question text)."""
    seen: set[tuple[str, str, str]] = set()
    out = []
    for r in pool:
        if not r["is_knowledge"]:
            continue
        key = (r["source_dataset"], _image_identity(r), r["question"].lower().strip())
        if key in seen:
            continue
        seen.add(key)
        out.append({**r, "knowledge_subtype": knowledge_subtype(r["question"])})
    return out


def select_side_records(pool: list[dict], n_per_side: int = 60, n_both: int = 20,
                        max_per_question_per_side: int = 8, seed: int = 0) -> list[dict]:
    """Open questions whose short gold answer names a side, balanced left/right.

    `max_per_question_per_side` stops one phrasing ("Where is the abnormality
    in this image?") from dominating a side.
    """
    candidates: dict[str, list[dict]] = {"left": [], "right": [], "both": []}
    for r in pool:
        if r["answer_format"] != "open" or r["is_knowledge"]:
            continue
        gold = r["gold_answer"]
        side = side_of(gold)
        if side is None or len(gold.split()) > 4:
            continue
        candidates[side].append({**r, "gold_side": side,
                                 "side_granularity": "side_only" if len(gold.split()) == 1 else "region"})

    rng = random.Random(seed)
    chosen: list[dict] = []
    for side, limit in (("left", n_per_side), ("right", n_per_side), ("both", n_both)):
        rows = candidates[side][:]
        rng.shuffle(rows)
        per_q: dict[str, int] = {}
        taken = []
        for r in rows:
            k = r["question"].lower().strip()
            if per_q.get(k, 0) >= max_per_question_per_side:
                continue
            per_q[k] = per_q.get(k, 0) + 1
            taken.append(r)
            if len(taken) >= limit:
                break
        chosen += taken
    rng.shuffle(chosen)
    return chosen


def split_dev_heldout(records: list[dict], dev_fraction: float = 0.3) -> tuple[list[dict], list[dict]]:
    """Deterministic split by image so no image appears on both sides.

    Retrieval design (what to retrieve, how many passages, prompt wording) may
    be tuned on the dev part only; the held-out part is scored once, at the end.
    Without this, a small test set gets tuned until it shows an improvement.
    """
    dev, held = [], []
    for r in records:
        bucket = int(hashlib.md5(_image_identity(r).encode()).hexdigest(), 16) % 1000
        (dev if bucket < dev_fraction * 1000 else held).append(r)
    return dev, held


def attach_images(records: list[dict], images_dir: str | Path) -> list[dict]:
    slake = download_slake_images([r for r in records if r["source_dataset"] == "SLAKE"], images_dir)
    vqa = download_vqa_rad_images([r for r in records if r["source_dataset"] == "VQA-RAD"], images_dir)
    for r in records:
        paths = slake if r["source_dataset"] == "SLAKE" else vqa
        r["image_path"] = paths[r["image_key"]]
        r["id"] = r["uid"]
    return records


def describe_set(records: list[dict]) -> dict:
    """Counts used in the documentation of how each set is composed."""
    def count(key):
        out: dict[str, int] = {}
        for r in records:
            out[str(r.get(key))] = out.get(str(r.get(key)), 0) + 1
        return dict(sorted(out.items()))

    return {
        "n": len(records),
        "by_source": count("source_dataset"),
        "by_split": count("source_split"),
        "by_subtype": count("knowledge_subtype") if records and "knowledge_subtype" in records[0] else None,
        "by_gold_side": count("gold_side") if records and "gold_side" in records[0] else None,
        "unique_questions": len({r["question"].lower().strip() for r in records}),
        "unique_gold_answers": len({r["gold_answer"].lower().strip() for r in records}),
        "unique_images": len({_image_identity(r) for r in records}),
    }


def build_knowledge_test_set(images_dir: str | Path, include_train: bool = True,
                             cache_path: str | Path | None = None, pool: list[dict] | None = None) -> list[dict]:
    """Pass `pool` (from `fetch_full_pool`) to build several sets from one fetch."""
    pool = pool if pool is not None else fetch_full_pool(include_train=include_train, cache_path=cache_path)
    return attach_images(select_knowledge_records(pool), images_dir)


def build_side_test_set(images_dir: str | Path, n_per_side: int = 60, n_both: int = 20, seed: int = 0,
                        include_train: bool = True, cache_path: str | Path | None = None,
                        pool: list[dict] | None = None) -> list[dict]:
    pool = pool if pool is not None else fetch_full_pool(include_train=include_train, cache_path=cache_path)
    return attach_images(select_side_records(pool, n_per_side=n_per_side, n_both=n_both, seed=seed), images_dir)


def main() -> None:
    import argparse

    parser = argparse.ArgumentParser(description="Build the knowledge and side test sets (no GPU needed).")
    parser.add_argument("--out", default="data")
    parser.add_argument("--images-dir", default="data/images")
    parser.add_argument("--skip-images", action="store_true", help="metadata only, no image download")
    args = parser.parse_args()

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    pool = fetch_full_pool(cache_path=out / "cache" / "pools_full.json")
    sets = {"knowledge": select_knowledge_records(pool), "side": select_side_records(pool)}
    summary = {}
    for name, records in sets.items():
        if not args.skip_images:
            attach_images(records, args.images_dir)
        else:
            for r in records:
                r["id"] = r["uid"]
        with open(out / f"{name}_test_set.jsonl", "w") as f:
            for r in records:
                f.write(json.dumps(r) + "\n")
        summary[name] = describe_set(records)
        print(name, json.dumps(summary[name], indent=2))
    (out / "extra_sets_summary.json").write_text(json.dumps(summary, indent=2))


In [ ]:
"""Scoring and review tooling for knowledge-type questions (symptoms, treatment,
prevention, cause, function, 'why'/'how can you tell').

Why this exists: on these questions the gold answer is usually a short list
("Chest pain, dyspnea"), and exact match scores a correct paraphrase
("Chest tightness, shortness of breath" -> shares 'dyspnea') as 0 while
token F1 can reward unrelated shared words. Two additions:

1. `item_scores`: splits the gold and predicted answers into items, maps a small
   hand-written table of synonyms onto one spelling, and measures how many gold
   items the prediction covers (recall) and how many predicted items are
   supported by the gold (precision). The synonym table is a convenience, not
   a clinically validated ontology. It is short enough to read in full.
2. A blind manual review sheet: baseline and retrieval answers are shuffled
   into columns the reviewer cannot attribute to a system, rated 0/1/2, and
   unblinded afterwards. Automated scores are a screening tool; this is the
   check that decides whether an improvement is real.
"""


import csv
import random
import re
from dataclasses import dataclass


# canonical spelling -> variants that mean the same thing (all lowercase).
CONCEPT_SYNONYMS: dict[str, list[str]] = {
    "dyspnea": ["shortness of breath", "short of breath", "difficulty breathing",
                "difficulty in breathing", "breathing difficulty", "breathlessness", "dyspnoea"],
    "hemoptysis": ["coughing up blood", "coughing blood", "bloody sputum", "blood in sputum"],
    "cough": ["coughing"],
    "chest pain": ["chest discomfort"],
    "expectoration": ["sputum production", "phlegm"],
    "fatigue": ["tiredness"],
    "fever": ["pyrexia"],
    "medication": ["medical treatment", "medical therapy", "drug therapy", "drug treatment", "medicine"],
    "surgery": ["surgical treatment", "surgical therapy", "operation", "surgical"],
}

_STOPWORDS = {"the", "a", "an", "of", "in", "on", "and", "to", "is", "are", "with", "by",
              "for", "can", "be", "it", "this", "that", "or", "as", "at"}

_VARIANT_TO_CANON: list[tuple[re.Pattern, str]] = sorted(
    ((re.compile(rf"\b{re.escape(v)}\b"), canon)
     for canon, variants in CONCEPT_SYNONYMS.items() for v in variants),
    key=lambda item: -len(item[0].pattern),
)

_SPLIT_RE = re.compile(r"[,;/]|\band\b|\bor\b", re.IGNORECASE)

SCORER_VERSION = 2  # 1: no stemming (used for the first knowledge-set report); 2: light stemming

RATING_RUBRIC = (
    "2 = correct (same meaning as the gold answer, wording may differ)\n"
    "1 = partly correct (some of the gold content, or correct but incomplete / with wrong extras)\n"
    "0 = wrong, unrelated, contradicts the gold answer, or does not answer the question"
)


def canonicalize(text: str) -> str:
    t = normalize(str(text))
    for pattern, canon in _VARIANT_TO_CANON:
        t = pattern.sub(canon, t)
    return t


def _stem(word: str) -> str:
    """Very light suffix stripping so that 'keeping' matches 'keep' and 'lungs'
    matches 'lung'. Added in scorer version 2 after 'keeping healthy' scored 0
    against the gold 'Keep healthy'. Deliberately crude; applied identically to
    gold and predicted text."""
    if len(word) > 5 and word.endswith("ing"):
        return word[:-3]
    if len(word) > 3 and word.endswith("s") and not word.endswith("ss"):
        return word[:-1]
    return word


def content_tokens(text: str) -> frozenset[str]:
    return frozenset(_stem(w) for w in canonicalize(text).split() if w not in _STOPWORDS)


def split_items(answer: str) -> list[str]:
    return [p.strip() for p in _SPLIT_RE.split(str(answer)) if p.strip()]


def _items_match(a: frozenset[str], b: frozenset[str]) -> bool:
    return bool(a) and bool(b) and (a <= b or b <= a)


@dataclass(frozen=True)
class ItemScores:
    recall: float
    precision: float
    f1: float


def item_scores(prediction: str, gold: str) -> ItemScores:
    """Item-level recall / precision / F1 after synonym canonicalization."""
    gold_items = [content_tokens(i) for i in split_items(gold)]
    pred_items = [content_tokens(i) for i in split_items(prediction)]
    gold_all, pred_all = content_tokens(gold), content_tokens(prediction)
    gold_items, pred_items = [g for g in gold_items if g], [p for p in pred_items if p]
    if not gold_items or not pred_items:
        return ItemScores(0.0, 0.0, 0.0)

    covered = sum(1 for g in gold_items if g <= pred_all or any(_items_match(g, p) for p in pred_items))
    supported = sum(1 for p in pred_items if p <= gold_all or any(_items_match(p, g) for g in gold_items))
    recall, precision = covered / len(gold_items), supported / len(pred_items)
    f1 = 0.0 if recall + precision == 0 else 2 * recall * precision / (recall + precision)
    return ItemScores(recall, precision, f1)


def score_knowledge_rows(rows: list[dict]) -> list[dict]:
    """Add item_recall / item_precision / item_f1 / token_f1 to each row."""
    out = []
    for r in rows:
        s = item_scores(r["prediction"], r["gold_answer"])
        out.append({**r, "item_recall": round(s.recall, 3), "item_precision": round(s.precision, 3),
                    "item_f1": round(s.f1, 3), "token_f1": round(token_f1(r["prediction"], r["gold_answer"]), 3)})
    return out


def summarize_knowledge(rows: list[dict]) -> dict[str, dict[str, float]]:
    """Mean scores overall and per knowledge_subtype for rows from `score_knowledge_rows`."""
    def mean(rs, key):
        return sum(float(r[key]) for r in rs) / len(rs) if rs else float("nan")

    groups: dict[str, list[dict]] = {"all": rows}
    for r in rows:
        groups.setdefault(r.get("knowledge_subtype", "unknown"), []).append(r)
    return {name: {"n": len(rs), "item_recall": mean(rs, "item_recall"), "item_f1": mean(rs, "item_f1"),
                   "token_f1": mean(rs, "token_f1"), "any_item_hit": sum(float(r["item_recall"]) > 0 for r in rs) / len(rs)}
            for name, rs in groups.items()}


def format_knowledge_summary(summary: dict[str, dict[str, float]]) -> str:
    L = ["=== Knowledge questions: item-level scores ===",
         f"{'subtype':>14} {'n':>4}  {'item_recall':>11}  {'item_f1':>7}  {'token_f1':>8}  {'>=1 item right':>14}"]
    for name, s in sorted(summary.items(), key=lambda kv: (kv[0] != "all", kv[0])):
        L.append(f"{name:>14} {s['n']:>4}  {s['item_recall']:>11.2f}  {s['item_f1']:>7.2f}  "
                 f"{s['token_f1']:>8.2f}  {s['any_item_hit']:>14.0%}")
    return "\n".join(L)


def make_blind_review(baseline_rows: list[dict], candidate_rows: list[dict], seed: int = 0,
                      candidate_name: str = "candidate") -> tuple[list[dict], list[dict]]:
    """Pair two systems' answers by id, randomize which one appears as
    answer_1, and return (sheet, key). Give the reviewer the sheet only; keep
    the key until all ratings are filled in."""
    cand = {r["id"]: r for r in candidate_rows}
    rng = random.Random(seed)
    sheet, key = [], []
    for b in baseline_rows:
        c = cand.get(b["id"])
        if c is None:
            continue
        flip = rng.random() < 0.5
        first, second = (c, b) if flip else (b, c)
        sheet.append({"id": b["id"], "question": b["question"], "gold_answer": b["gold_answer"],
                      "answer_1": first["prediction"], "answer_2": second["prediction"],
                      "rating_1": "", "rating_2": "", "notes": ""})
        key.append({"id": b["id"], "answer_1_is": candidate_name if flip else "baseline"})
    return sheet, key


def write_csv(path: str, rows: list[dict]) -> None:
    if not rows:
        return
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0]))
        w.writeheader()
        w.writerows(rows)


def summarize_blind_review(sheet: list[dict], key: list[dict], candidate_name: str = "candidate") -> dict:
    """Unblind completed ratings: mean rating per system, and wins / losses /
    ties for the candidate with an exact sign test (ties excluded)."""
    key_by_id = {k["id"]: k["answer_1_is"] for k in key}
    ratings = {"baseline": [], candidate_name: []}
    wins = losses = ties = 0
    for row in sheet:
        if str(row["rating_1"]).strip() == "" or str(row["rating_2"]).strip() == "":
            continue
        r1, r2 = int(row["rating_1"]), int(row["rating_2"])
        if key_by_id[row["id"]] == candidate_name:
            cand_r, base_r = r1, r2
        else:
            cand_r, base_r = r2, r1
        ratings[candidate_name].append(cand_r)
        ratings["baseline"].append(base_r)
        wins += cand_r > base_r
        losses += cand_r < base_r
        ties += cand_r == base_r
    n = len(ratings["baseline"])
    return {
        "n_rated": n,
        "mean_baseline": sum(ratings["baseline"]) / n if n else float("nan"),
        f"mean_{candidate_name}": sum(ratings[candidate_name]) / n if n else float("nan"),
        "wins": wins, "losses": losses, "ties": ties,
        "sign_test_p": binom_two_sided(wins, wins + losses, 0.5) if wins + losses else float("nan"),
    }


def constant_floor(rows: list[dict]) -> dict[str, float]:
    """Item recall of a system that ignores the image and always writes the most common gold answer for the
    question type (leave-one-out, so a question's own answer is never used). Amendment 2 of the protocol.
    rows need: knowledge_subtype, gold_answer. Returns {'all': ..., subtype: ...}."""
    from collections import Counter
    by_sub: dict[str, list[int]] = {}
    for i, r in enumerate(rows):
        by_sub.setdefault(r["knowledge_subtype"], []).append(i)
    scores: dict[str, list[float]] = {}
    for sub, idx in by_sub.items():
        for i in idx:
            others = Counter(rows[j]["gold_answer"] for j in idx if j != i)
            if not others:
                continue
            best = sorted(others.items(), key=lambda kv: (-kv[1], kv[0]))[0][0]
            scores.setdefault(sub, []).append(item_scores(best, rows[i]["gold_answer"]).recall)
    out = {sub: sum(v) / len(v) for sub, v in scores.items()}
    flat = [x for v in scores.values() for x in v]
    out["all"] = sum(flat) / len(flat) if flat else float("nan")
    return out


def distinct_answers(rows: list[dict], key: str = "prediction") -> dict[str, int]:
    """Number of different answers (after normalisation) per knowledge subtype and overall."""
    seen: dict[str, set] = {}
    for r in rows:
        seen.setdefault(r["knowledge_subtype"], set()).add(normalize(str(r[key])))
    out = {sub: len(v) for sub, v in seen.items()}
    out["all"] = len({normalize(str(r[key])) for r in rows})
    return out


In [ ]:
"""Disease labels for SLAKE chest X-rays, and scoring for "can the model name the disease?".

Why this exists (docs/phase2_rag_plan.md): SLAKE's knowledge answers (cause, symptoms, treatment,
prevention) are fixed by the disease on the image, and the question text never names it. SLAKE
states the disease in a separate question on the same image ("What diseases are included in the
picture?"). Those answers are used here for two things only:

1. Scoring a disease-naming test (Phase 1d): the label is the answer key.
2. "Oracle" arms in Phase 2: the true disease is handed to the system to measure an upper bound.
   Oracle arms are never counted as the project's result (protocol rule 3, Amendment 5).

The candidate vocabulary shown to the model is the public ChestX-ray14 pathology list, not the
list of diseases found in SLAKE.
"""


import random
import re
from collections import Counter, defaultdict


CHESTXRAY14 = [
    "Atelectasis", "Cardiomegaly", "Consolidation", "Edema", "Effusion", "Emphysema", "Fibrosis",
    "Hernia", "Infiltration", "Mass", "Nodule", "Pleural Thickening", "Pneumonia", "Pneumothorax",
]
NO_FINDING = "No Finding"
CANDIDATES = CHESTXRAY14 + [NO_FINDING]

# SLAKE spelling (lowercase) -> canonical name. Includes the dataset's typos ("Noudle", "Cardiomega").
SPELLING_MAP = {
    "pulmonary mass": "Mass", "mass": "Mass",
    "pulmonary nodule": "Nodule", "nodule": "Nodule", "noudle": "Nodule",
    "pleural effusion": "Effusion", "effusion": "Effusion",
    "pulmonary infiltration": "Infiltration", "infiltration": "Infiltration",
    "cardiomegaly": "Cardiomegaly", "cardiomegal": "Cardiomegaly", "cardiomega": "Cardiomegaly",
    "atelectasis": "Atelectasis", "pneumonia": "Pneumonia", "pneumothorax": "Pneumothorax",
}

# Phrases in free-text model answers that count as naming a disease.
_PARSE_PATTERNS: list[tuple[str, re.Pattern]] = [
    ("Cardiomegaly", re.compile(r"cardiomegaly|enlarged heart|heart (?:is )?enlarged|enlarged cardiac")),
    ("Atelectasis", re.compile(r"atelecta")),
    ("Consolidation", re.compile(r"consolidat")),
    ("Edema", re.compile(r"\bedema\b|\boedema\b")),
    ("Effusion", re.compile(r"effusion")),
    ("Emphysema", re.compile(r"emphysema")),
    ("Fibrosis", re.compile(r"fibrosis|fibrotic")),
    ("Hernia", re.compile(r"hernia")),
    ("Infiltration", re.compile(r"infiltrat")),
    ("Mass", re.compile(r"\bmass(?:es)?\b")),
    ("Nodule", re.compile(r"\bnodul")),
    ("Pleural Thickening", re.compile(r"pleural thickening")),
    ("Pneumonia", re.compile(r"pneumonia")),
    ("Pneumothorax", re.compile(r"pneumothorax")),
    (NO_FINDING, re.compile(r"no finding|no acute|no abnormal|normal chest|no disease|no evidence of")),
]

_DISEASE_Q = re.compile(r"^\s*what diseases?\b", re.IGNORECASE)
_SPLIT_LABELS = re.compile(r"[,;/]|\band\b")


def is_disease_naming_question(question: str) -> bool:
    return bool(_DISEASE_Q.match(question))


def canonical_labels(answer: str) -> tuple[str, ...]:
    """'Cardiomegaly, Infiltration, Effusion' -> ('Cardiomegaly', 'Effusion', 'Infiltration').
    Unknown names are kept (title-cased) so nothing is silently dropped."""
    out = set()
    for part in _SPLIT_LABELS.split(str(answer)):
        p = part.strip().lower()
        if p:
            out.add(SPELLING_MAP.get(p, p.title()))
    return tuple(sorted(out))


def build_image_labels(pool: list[dict]) -> dict[str, dict]:
    """image_key -> {'labels': tuple, 'split': str}, from SLAKE disease-naming questions.
    If an image has several such questions the labels are merged."""
    merged: dict[str, dict] = {}
    for r in pool:
        if r["source_dataset"] != "SLAKE" or not is_disease_naming_question(r["question"]):
            continue
        entry = merged.setdefault(r["image_key"], {"labels": set(), "split": r["source_split"]})
        entry["labels"].update(canonical_labels(r["gold_answer"]))
    return {k: {"labels": tuple(sorted(v["labels"])), "split": v["split"]} for k, v in merged.items()}


def disease_id_items(pool: list[dict]) -> list[dict]:
    """One item per labelled SLAKE chest image, ordered by image key. Needs `attach_images` afterwards."""
    labels = build_image_labels(pool)
    keys = {r["image_key"]: r for r in pool if r["source_dataset"] == "SLAKE" and r["image_key"] in labels}
    items = []
    for key in sorted(labels):
        r = keys[key]
        items.append({
            "id": f"DIS_{key.split('/')[0]}", "uid": f"DIS_{key.split('/')[0]}", "source_dataset": "SLAKE",
            "source_split": labels[key]["split"], "image_key": key, "question": "(disease naming)",
            "gold_answer": ", ".join(labels[key]["labels"]), "labels": list(labels[key]["labels"]),
        })
    return items


def attach_oracle_disease(knowledge_records: list[dict], pool: list[dict]) -> list[dict]:
    """Add `oracle_disease` (single canonical name, or None) to every knowledge record. Multi-label
    images, VQA-RAD rows and images without a disease question get None. Oracle use only."""
    labels = build_image_labels(pool)
    out = []
    for r in knowledge_records:
        lab = labels.get(r["image_key"]) if r["source_dataset"] == "SLAKE" else None
        out.append({**r, "oracle_disease": lab["labels"][0] if lab and len(lab["labels"]) == 1 else None})
    return out


def parse_diseases(text: str) -> list[str]:
    """Diseases named in a free-text answer, in order of first appearance."""
    t = str(text).lower()
    found = [(m.start(), name) for name, pat in _PARSE_PATTERNS if (m := pat.search(t))]
    return [name for _, name in sorted(found)]


# ------------------------------------------------------------------------ prompts

DISEASE_PROMPTS = {
    "open": "What disease is shown in this chest X-ray?",
    "forced": ("Which of these is present in this chest X-ray: " + ", ".join(CANDIDATES) + "? "
               "Answer with one name from the list."),
}
PERCLASS_PROMPT = "{name} Answer yes or no."
_PERCLASS_NAMES = {"Effusion": "pleural effusion", "Mass": "a mass", "Nodule": "a nodule", "Edema": "pulmonary edema",
                   "Fibrosis": "pulmonary fibrosis", "Hernia": "a hernia", "Infiltration": "infiltration"}


def perclass_question(name: str) -> str:
    """Yes/no question for one candidate. 'No Finding' is asked as 'Is this chest X-ray normal?'."""
    if name == NO_FINDING:
        return PERCLASS_PROMPT.format(name="Is this chest X-ray normal?")
    return PERCLASS_PROMPT.format(name=f"Is there {_PERCLASS_NAMES.get(name, name.lower())} in this chest X-ray?")


# ---------------------------------------------------------------------- scoring

def auc_one_vs_rest(scores: list[float], positives: list[bool]) -> float:
    """Probability that a random positive outranks a random negative (ties count half)."""
    pos = [s for s, p in zip(scores, positives) if p]
    neg = [s for s, p in zip(scores, positives) if not p]
    if not pos or not neg:
        return float("nan")
    wins = sum((a > b) + 0.5 * (a == b) for a in pos for b in neg)
    return wins / (len(pos) * len(neg))


def _any_match(pred: list[str], gold: list[str]) -> bool:
    return bool(set(pred) & set(gold))


def _strict_match(pred: list[str], gold: list[str]) -> bool:
    return len(pred) == 1 and set(pred) == set(gold)


def disease_report(rows: list[dict], n_shuffles: int = 2000, seed: int = 0) -> dict:
    """rows need: labels (gold list), pred (list of predicted names; first is the top choice),
    split. Reports accuracy on the top choice and with any named disease, against
    always-the-most-common-label and against shuffled labels."""
    n = len(rows)
    top = [r["pred"][:1] for r in rows]
    gold = [r["labels"] for r in rows]
    acc_top = sum(_any_match(p, g) for p, g in zip(top, gold))
    acc_any = sum(_any_match(r["pred"], r["labels"]) for r in rows)
    acc_strict = sum(_strict_match(r["pred"], r["labels"]) for r in rows)

    counts = Counter(l for g in gold for l in set(g))
    majority = counts.most_common(1)[0][0] if counts else None
    maj_acc = sum(majority in g for g in gold) if majority else 0

    rng = random.Random(seed)
    shuffled = []
    for _ in range(n_shuffles):
        perm = gold[:]
        rng.shuffle(perm)
        shuffled.append(sum(_any_match(p, g) for p, g in zip(top, perm)))
    shuffled.sort()

    per_class: dict[str, dict] = {}
    for c in sorted(counts):
        idx = [i for i, g in enumerate(gold) if c in g]
        per_class[c] = {"n": len(idx), "recall": sum(c in top[i] for i in idx) / len(idx),
                        "predicted": sum(c in t for t in top)}
    by_split = {}
    for sp in sorted({r["split"] for r in rows}):
        sub = [r for r in rows if r["split"] == sp]
        by_split[sp] = {"n": len(sub), "acc_top": sum(_any_match(r["pred"][:1], r["labels"]) for r in sub)}
    return {
        "n": n, "acc_top": acc_top / n if n else float("nan"), "acc_top_ci": wilson_interval(acc_top, n),
        "acc_any": acc_any / n if n else float("nan"), "acc_strict": acc_strict / n if n else float("nan"),
        "unparsed": sum(not r["pred"] for r in rows),
        "majority_label": majority, "majority_acc": maj_acc / n if n else float("nan"),
        "p_vs_majority": binom_two_sided(acc_top, n, maj_acc / n) if n and maj_acc else float("nan"),
        "shuffled_mean": sum(shuffled) / len(shuffled) / n if n else float("nan"),
        "shuffled_p95": shuffled[int(0.95 * len(shuffled)) - 1] / n if n else float("nan"),
        "per_class": per_class, "by_split": by_split,
    }


def format_disease_report(by_method: dict[str, dict]) -> str:
    L = ["=== Disease naming (top choice counts as correct if it matches any gold label) ===",
         f"{'method':>16} {'n':>4} {'acc':>6} {'95% CI':>11} {'any named':>9} {'strict':>7} {'majority':>9} "
         f"{'shuffled':>9} {'p vs maj':>9} {'unparsed':>8}"]
    for name, r in by_method.items():
        lo, hi = r["acc_top_ci"]
        L.append(f"{name:>16} {r['n']:>4} {r['acc_top']:>6.0%} {lo:>5.0%}-{hi:<5.0%} {r['acc_any']:>9.0%} "
                 f"{r['acc_strict']:>7.0%} {r['majority_acc']:>9.0%} {r['shuffled_mean']:>9.0%} "
                 f"{r['p_vs_majority']:>9.3f} {r['unparsed']:>8}")
    L.append("majority = always answering the most common label; shuffled = same answers against shuffled gold labels.")
    for name, r in by_method.items():
        L.append(f"\n[{name}] per class (n, recall, times predicted):")
        L.append("   " + "; ".join(f"{c} {v['n']}, {v['recall']:.0%}, {v['predicted']}" for c, v in r["per_class"].items()))
        L.append("   by split: " + "; ".join(f"{sp} {v['acc_top']}/{v['n']}" for sp, v in r["by_split"].items()))
    return "\n".join(L)


def disease_reading(rep: dict) -> str:
    """The pre-stated reading (docs/phase2_rag_plan.md, section 3) for one method's report."""
    above = (rep["acc_top"] > rep["majority_acc"] and rep["p_vs_majority"] < 0.05
             and rep["acc_top"] > rep["shuffled_p95"])
    lo, hi = rep["acc_top_ci"]
    head = (f"top-choice accuracy {rep['acc_top']:.0%} (95% CI {lo:.0%}-{hi:.0%}) against majority "
            f"{rep['majority_acc']:.0%} (p = {rep['p_vs_majority']:.3f}) and shuffled 95th percentile "
            f"{rep['shuffled_p95']:.0%}")
    if above:
        return head + ": above chance by both criteria."
    return head + ": NOT shown to be above chance. Perception is the bottleneck for naming the disease."


def choose_method(by_method_validation: dict[str, dict]) -> str:
    """The method used downstream: highest accuracy on the validation images; ties go to the
    order of the dict (open, forced, perclass), so the choice is fixed before looking at test rows."""
    return max(by_method_validation, key=lambda m: by_method_validation[m]["acc_top"])


def perclass_predictions(scores: dict[str, dict[str, float]]) -> dict[str, list[str]]:
    """scores: image id -> {class: yes-minus-no logit}. Predicted list = classes ordered by score."""
    return {i: sorted(s, key=lambda c: -s[c]) for i, s in scores.items()}


def class_table(scores: dict[str, dict[str, float]], labels: dict[str, list[str]]) -> dict[str, float]:
    """One-vs-rest AUC per class over all images that have scores."""
    ids = sorted(scores)
    out = {}
    for c in sorted({c for s in scores.values() for c in s}):
        pos = [c in labels[i] for i in ids]
        out[c] = auc_one_vs_rest([scores[i][c] for i in ids], pos)
    return out


def group_by_label(items: list[dict]) -> dict[str, list[dict]]:
    g = defaultdict(list)
    for it in items:
        for l in it["labels"]:
            g[l].append(it)
    return dict(g)


In [ ]:
"""Retrieval for Phase 2: corpus loading, BM25, query and prompt construction, retrieval scoring.

Design and rules: docs/phase2_rag_plan.md and Amendment 5 in docs/evaluation_protocol.md.

Corpora (both independent of SLAKE; see the plan, section 8):
  * MedRAG textbooks on Hugging Face (18 textbooks chunked to <= 1000 characters). Research use only,
    downloaded at run time, never committed.
  * MedlinePlus Health Topics (National Library of Medicine, public domain), split into passages at
    each section heading ("What causes pneumonia?").

BM25 is implemented here (about 40 lines, no dependencies) so that the index is auditable and the
unit tests do not need extra packages.
"""


import html
import json
import math
import re
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

import requests


RAG_CACHE_DIR = Path("rag_cache")

MEDRAG_TEXTBOOK_URL = "https://huggingface.co/datasets/MedRAG/textbooks/resolve/main/chunk/{name}.jsonl"
MEDRAG_TEXTBOOK_NAMES = [
    "Anatomy_Gray", "Biochemistry_Lippincott", "Cell_Biology_Alberts", "First_Aid_Step1", "First_Aid_Step2",
    "Gynecology_Novak", "Histology_Ross", "Immunology_Janeway", "InternalMed_Harrison", "Neurology_Adams",
    "Obstentrics_Williams", "Pathology_Robbins", "Pathoma_Husain", "Pediatrics_Nelson", "Pharmacology_Katzung",
    "Physiology_Levy", "Psichiatry_DSM-5", "Surgery_Schwartz",
]
MEDLINEPLUS_INDEX_URL = "https://medlineplus.gov/xml.html"
MEDLINEPLUS_XML_URL = "https://medlineplus.gov/xml/{name}"

# A textbook chunk is kept only if it mentions a chest disease or a chest structure. This keeps the index
# small enough to build in about a minute. It uses the ChestX-ray14 names, not SLAKE's labels.
CHEST_FILTER = re.compile(
    r"atelecta|cardiomegal|cardiac enlargement|consolidat|\bedema\b|effusion|emphysema|fibrosis|hernia|"
    r"infiltrat|\bmass(?:es)?\b|nodul|pleur|pneumonia|pneumothora|lung|pulmonary|bronch|thora|\bchest\b|"
    r"cardiac|heart|cardio", re.IGNORECASE)

# ------------------------------------------------------------------------------------ BM25


def tokenize(text: str) -> list[str]:
    return [_stem(w) for w in re.findall(r"[a-z0-9]+", str(text).lower()) if w not in _STOPWORDS]


class BM25:
    """Okapi BM25 over a list of passages ({'text', 'title', ...}); title and text both count."""

    def __init__(self, passages: list[dict], k1: float = 1.5, b: float = 0.75):
        self.passages, self.k1, self.b = passages, k1, b
        self.postings: dict[str, list[tuple[int, int]]] = defaultdict(list)
        self.doc_len: list[int] = []
        for i, p in enumerate(passages):
            toks = tokenize(f"{p.get('title', '')} {p['text']}")
            self.doc_len.append(len(toks))
            for term, tf in Counter(toks).items():
                self.postings[term].append((i, tf))
        self.n = len(passages)
        self.avg_len = sum(self.doc_len) / self.n if self.n else 0.0

    def idf(self, term: str) -> float:
        df = len(self.postings.get(term, ()))
        return math.log(1 + (self.n - df + 0.5) / (df + 0.5))

    def search(self, query: str, k: int = 3, topics: set[str] | None = None) -> list[tuple[float, dict]]:
        """`topics`, when given, restricts the search to passages whose `topic` field is in the set."""
        scores: dict[int, float] = defaultdict(float)
        # A term repeated in the query counts proportionally more, which is how a query puts extra weight on
        # the disease name (build_query, disease_weight).
        for term, qtf in Counter(tokenize(query)).items():
            idf = self.idf(term)
            for i, tf in self.postings.get(term, ()):
                denom = tf + self.k1 * (1 - self.b + self.b * self.doc_len[i] / self.avg_len)
                if topics is None or self.passages[i].get("topic") in topics:
                    scores[i] += qtf * idf * tf * (self.k1 + 1) / denom
        ranked = sorted(scores.items(), key=lambda kv: (-kv[1], kv[0]))[:k]
        return [(s, self.passages[i]) for i, s in ranked]


# --------------------------------------------------------------------------------- corpora


def load_textbook_passages(cache_dir: str | Path = RAG_CACHE_DIR, chest_only: bool = True) -> list[dict]:
    """MedRAG textbook chunks, optionally restricted to chest-related chunks (CHEST_FILTER)."""
    cache_dir = Path(cache_dir)
    out_path = cache_dir / ("textbooks_chest.jsonl" if chest_only else "textbooks_all.jsonl")
    if out_path.exists() and out_path.stat().st_size > 0:
        return [json.loads(line) for line in out_path.read_text().splitlines() if line]
    passages = []
    for name in MEDRAG_TEXTBOOK_NAMES:
        path = _download_file(MEDRAG_TEXTBOOK_URL.format(name=name), cache_dir / "raw" / f"{name}.jsonl")
        for line in path.read_text().splitlines():
            if not line.strip():
                continue
            row = json.loads(line)
            text = row.get("content") or row.get("contents") or ""
            if chest_only and not CHEST_FILTER.search(text):
                continue
            passages.append({"source": "textbooks", "id": row.get("id", ""), "title": row.get("title", name),
                             "text": " ".join(text.split())})
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text("\n".join(json.dumps(p) for p in passages))
    return passages


def discover_medlineplus_xml_name(timeout: int = 60) -> str:
    """Name of the newest bulk file listed on MedlinePlus's XML page (the file name carries a date)."""
    page = requests.get(MEDLINEPLUS_INDEX_URL, timeout=timeout).text
    names = sorted(set(re.findall(r"mplus_topics_\d{4}-\d{2}-\d{2}\.xml", page)))
    if not names:
        raise RuntimeError("no mplus_topics_*.xml file listed on " + MEDLINEPLUS_INDEX_URL)
    return names[-1]


def _clean_html(fragment: str) -> str:
    t = re.sub(r"</li>", "; ", fragment)
    t = re.sub(r"<[^>]+>", " ", t)
    return " ".join(html.unescape(t).replace("\xa0", " ").split())


def medlineplus_passages_from_xml(xml_text: str) -> list[dict]:
    """One passage per section of each English health topic. Section = text under one <h3> heading."""
    root = ET.fromstring(xml_text)
    out = []
    for topic in root:
        if topic.attrib.get("language") != "English":
            continue
        title = topic.attrib.get("title", "")
        summary = topic.findtext("full-summary") or ""
        parts = re.split(r"(?i)<h3>(.*?)</h3>", summary)
        sections = [("", parts[0])] + [(parts[i], parts[i + 1]) for i in range(1, len(parts) - 1, 2)]
        for heading, body in sections:
            text = _clean_html(body)
            if len(text) < 40:
                continue
            head = _clean_html(heading)
            out.append({"source": "medlineplus", "id": f"{topic.attrib.get('id', '')}:{head[:40]}",
                        "title": f"{title}: {head}" if head else title, "text": text,
                        "topic": title, "url": topic.attrib.get("url", "")})
    return out


def load_medlineplus_passages(cache_dir: str | Path = RAG_CACHE_DIR) -> list[dict]:
    cache_dir = Path(cache_dir)
    existing = sorted(cache_dir.glob("mplus_topics_*.xml"))
    if existing:
        path = existing[-1]
    else:
        name = discover_medlineplus_xml_name()
        path = _download_file(MEDLINEPLUS_XML_URL.format(name=name), cache_dir / name)
    return medlineplus_passages_from_xml(path.read_text(encoding="utf-8"))


def build_retrievers(cache_dir: str | Path = RAG_CACHE_DIR) -> dict[str, BM25]:
    """One index per corpus: {'textbooks': BM25, 'medlineplus': BM25}."""
    return {"textbooks": BM25(load_textbook_passages(cache_dir)),
            "medlineplus": BM25(load_medlineplus_passages(cache_dir))}


# ------------------------------------------------------------------- queries and prompts

# Words added to the disease name per question type. Chosen from how textbooks and MedlinePlus
# headings are worded ("What causes ...", "Symptoms", "Treatment"), before any result was seen.
SUBTYPE_QUERY_WORDS = {
    "cause": "causes etiology",
    "symptoms": "symptoms signs clinical presentation",
    "treatment": "treatment management therapy",
    "prevention": "prevention prevent risk factors",
}

# canonical ChestX-ray14 name -> words used in the query. Revised once after a dev-only diagnosis (experiment log,
# entry 14): the first version used the radiology name alone and missed diseases that reference texts file under
# another name (cardiomegaly under cardiomyopathy / enlarged heart, atelectasis under collapsed lung).
QUERY_NAMES = {
    "Atelectasis": "atelectasis collapsed lung", "Cardiomegaly": "cardiomegaly enlarged heart cardiomyopathy",
    "Consolidation": "lung consolidation pneumonia", "Edema": "pulmonary edema", "Effusion": "pleural effusion",
    "Emphysema": "emphysema", "Fibrosis": "pulmonary fibrosis", "Hernia": "hiatal hernia",
    "Infiltration": "pulmonary infiltrates", "Mass": "lung mass tumor cancer",
    "Nodule": "pulmonary nodule lung cancer", "Pleural Thickening": "pleural thickening",
    "Pneumonia": "pneumonia", "Pneumothorax": "pneumothorax collapsed lung",
}
# MedlinePlus health-topic titles that cover each disease, chosen from the list of topic titles and from medical
# knowledge, not from any answer in the test sets. Used by the "medlineplus_topic" source.
MEDLINEPLUS_TOPICS = {
    "Atelectasis": {"Collapsed Lung"}, "Cardiomegaly": {"Cardiomyopathy", "Heart Failure"},
    "Consolidation": {"Pneumonia"}, "Edema": {"Edema"}, "Effusion": {"Pleural Disorders"},
    "Emphysema": {"Emphysema"}, "Fibrosis": {"Pulmonary Fibrosis"}, "Hernia": {"Hiatal Hernia"},
    "Infiltration": {"Pneumonia", "Interstitial Lung Diseases"}, "Mass": {"Lung Cancer"},
    "Nodule": {"Lung Cancer"}, "Pleural Thickening": {"Pleural Disorders"}, "Pneumonia": {"Pneumonia"},
    "Pneumothorax": {"Collapsed Lung"},
}
HINT_NAMES = {
    "Atelectasis": "atelectasis", "Cardiomegaly": "cardiomegaly", "Consolidation": "consolidation",
    "Edema": "pulmonary edema", "Effusion": "pleural effusion", "Emphysema": "emphysema",
    "Fibrosis": "pulmonary fibrosis", "Hernia": "hiatal hernia", "Infiltration": "pulmonary infiltration",
    "Mass": "pulmonary mass", "Nodule": "pulmonary nodule", "Pleural Thickening": "pleural thickening",
    "Pneumonia": "pneumonia", "Pneumothorax": "pneumothorax",
}

STYLE_INSTRUCTION = "Answer with a short comma-separated list of 2 to 4 items."


def usable_disease(disease: str | None) -> bool:
    return bool(disease) and disease != NO_FINDING and disease in QUERY_NAMES


def build_query(question: str, subtype: str, disease: str | None, disease_weight: int = 1) -> str:
    """Disease plus question-type words when both are known; otherwise the question text itself.
    `disease_weight` repeats the disease name so that it outweighs the question-type words."""
    if usable_disease(disease) and subtype in SUBTYPE_QUERY_WORDS:
        name = " ".join([QUERY_NAMES[disease]] * max(1, disease_weight))
        return f"{name} {SUBTYPE_QUERY_WORDS[subtype]}"
    return question


def truncate_passage(text: str, max_chars: int) -> str:
    """Cut at a sentence end before max_chars when there is one in the second half, else at a word."""
    if len(text) <= max_chars:
        return text
    cut = text[:max_chars]
    stop = max(cut.rfind(". "), cut.rfind("? "), cut.rfind("; "))
    if stop > max_chars // 2:
        return cut[:stop + 1]
    return cut.rsplit(" ", 1)[0] + " ..."


def retrieve_passages(retriever: BM25, query: str, k: int, max_chars: int) -> list[dict]:
    hits = retriever.search(query, k=k)
    return [{**p, "score": round(s, 3), "text_used": truncate_passage(p["text"], max_chars)} for s, p in hits]


RETRIEVAL_SOURCES = ("textbooks", "medlineplus", "medlineplus_topic", "topic+textbooks")


def retrieve_context(retrievers: dict[str, BM25], source: str, question: str, subtype: str,
                     disease: str | None, k: int, max_chars: int, disease_weight: int = 1) -> list[dict]:
    """Passages for one question.
    textbooks / medlineplus: BM25 over that corpus with the query from `build_query`.
    medlineplus_topic: BM25 restricted to the MedlinePlus topics for the disease (plain BM25 when no disease).
    topic+textbooks: the best topic passage first, the rest from the textbooks."""
    query = build_query(question, subtype, disease, disease_weight)
    topics = MEDLINEPLUS_TOPICS.get(disease) if usable_disease(disease) else None

    def top(name: str, n: int, restrict=None) -> list[dict]:
        return [{**p, "score": round(sc, 3), "text_used": truncate_passage(p["text"], max_chars)}
                for sc, p in retrievers[name].search(query, k=n, topics=restrict)]

    if source in ("textbooks", "medlineplus"):
        return top(source, k)
    if source == "medlineplus_topic":
        return top("medlineplus", k, topics)
    if source == "topic+textbooks":
        return top("medlineplus", 1, topics) + (top("textbooks", k - 1) if k > 1 else [])
    raise ValueError(f"unknown source {source!r}")


def build_prompt(question: str, disease: str | None = None, passages: list[dict] | None = None) -> str:
    """The prompt for every Phase 2 arm. No disease and no passages gives the style-matched baseline (A1)."""
    parts = []
    if passages:
        parts.append("Reference: " + " ".join(p["text_used"] for p in passages))
    if usable_disease(disease):
        parts.append(f"The finding in this image is {HINT_NAMES[disease]}.")
    parts += [question, STYLE_INSTRUCTION]
    return "\n".join(parts)


# ----------------------------------------------------------------------- retrieval scoring

_SENT_SPLIT = re.compile(r"(?<=[.!?;])\s+")


def gold_item_coverage(passages: list[dict], gold_answer: str) -> dict[str, float]:
    """How much of the gold answer's item list the retrieved text holds, with no model involved.

    loose:  an item counts if all its content words appear anywhere in the retrieved text.
    strict: an item counts only if all its content words appear within one sentence.
    Both use the same synonym table and stemming as the answer scorer (scorer version 2)."""
    items = [content_tokens(i) for i in split_items(gold_answer)]
    items = [i for i in items if i]
    if not items:
        return {"loose": 0.0, "strict": 0.0}
    texts = [p.get("text_used", p["text"]) for p in passages]
    all_tokens = content_tokens(" ".join(texts))
    sentences = [content_tokens(s) for t in texts for s in _SENT_SPLIT.split(t)]
    loose = sum(i <= all_tokens for i in items) / len(items)
    strict = sum(any(i <= s for s in sentences) for i in items) / len(items)
    return {"loose": loose, "strict": strict}


# ------------------------------------------------------------- Stage 2: retrieval without a model

MAX_CONTEXT_CHARS = 1400  # about 350 tokens; the model's context is 2048 tokens including the image


def coverage_for_setting(records: list[dict], retrievers: dict[str, BM25], source: str, k: int, max_chars: int,
                         disease_weight: int = 1, disease_of=lambda r: r.get("oracle_disease")) -> list[dict]:
    """For each record: retrieve with the query built from `disease_of(record)` and score how much of the
    gold answer's item list the retrieved text holds. No model, no gold access except for scoring."""
    out = []
    for r in records:
        disease = disease_of(r)
        passages = retrieve_context(retrievers, source, r["question"], r["knowledge_subtype"], disease,
                                    k, max_chars, disease_weight)
        cov = gold_item_coverage(passages, r["gold_answer"])
        out.append({"id": r.get("id") or r.get("uid"), "subtype": r["knowledge_subtype"],
                    "query": build_query(r["question"], r["knowledge_subtype"], disease, disease_weight), **cov})
    return out


def mean_coverage(rows: list[dict]) -> dict[str, float]:
    n = len(rows)
    return {"n": n, "strict": sum(r["strict"] for r in rows) / n if n else float("nan"),
            "loose": sum(r["loose"] for r in rows) / n if n else float("nan")}


def choose_setting(results: dict[tuple, dict]) -> tuple:
    """results: (source, k, max_chars, disease_weight) -> mean_coverage on the dev questions.
    Highest strict coverage, then highest loose coverage, then the shortest context, then grid order."""
    return min(results, key=lambda s: (-results[s]["strict"], -results[s]["loose"], s[1] * s[2]))


In [ ]:
import json, os, random
from collections import Counter

QUICK = False                  # True = 16-question smoke test, then set back to False
DISEASE_METHOD = None          # <-- paste CHOSEN_METHOD from notebook 04: "open", "forced" or "perclass"
assert DISEASE_METHOD in ("open", "forced", "perclass"), "set DISEASE_METHOD from notebook 04 first"

# Frozen before any model run (docs/phase2_rag_plan.md, section 10)
SOURCE, K, MAX_CHARS, WEIGHT = "topic+textbooks", 2, 700, 1

pool = fetch_full_pool()
know_items = attach_images(attach_oracle_disease(select_knowledge_records(pool), pool), "images")
dev_ids = {r["id"] for r in split_dev_heldout(know_items)[0]}
for r in know_items:
    r["is_dev"] = r["id"] in dev_ids
    r["has_oracle"] = bool(r["oracle_disease"]) and r["knowledge_subtype"] != "function"
if QUICK:
    rng = random.Random(0)
    oracle = [r for r in know_items if r["has_oracle"]]
    others = [r for r in know_items if not r["has_oracle"]]
    know_items = rng.sample(oracle, 10) + rng.sample(others, 6)

print("QUICK mode:", QUICK, "| disease method:", DISEASE_METHOD)
print("Knowledge questions:", len(know_items), dict(Counter(r["knowledge_subtype"] for r in know_items)))
print("With a known single disease (oracle subset):", sum(r["has_oracle"] for r in know_items))
print("Retrieval:", SOURCE, "k =", K, "max chars =", MAX_CHARS)
retrievers = build_retrievers()
print("Passages indexed:", {k: v.n for k, v in retrievers.items()})


## 3. Load CheXagent-2-3b

Identical to the other notebooks (downloads about 12.6 GB the first time).

In [ ]:
import glob, re
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "StanfordAIMI/CheXagent-2-3b"
DEVICE = "cuda"
DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("Using dtype:", DTYPE)

_VERSION_ASSERT = re.compile(r'^assert transformers\.__version__ == "4\.40\.0".*$', flags=re.M)

def patch_version_asserts(model_dir):
    """Neutralize the model code's hard-coded transformers==4.40.0 checks.
    Returns the number of lines patched (expected: 2)."""
    patched = 0
    for path in glob.glob(f"{model_dir}/*.py"):
        text = open(path).read()
        new_text, n = _VERSION_ASSERT.subn("pass", text)
        if n:
            open(path, "w").write(new_text)
            patched += n
    return patched

_IMAGE_BRANCH_ORIGINAL = "if past_key_values is None and torch.any(input_ids == self.tokenizer.img_start_id):"
_IMAGE_BRANCH_PATCHED = (
    "if (past_key_values is None or (hasattr(past_key_values, 'get_seq_length') "
    "and past_key_values.get_seq_length() == 0)) and torch.any(input_ids == self.tokenizer.img_start_id):"
)

def patch_image_cache_check(model_dir):
    """transformers >= 4.45 passes an empty DynamicCache (not None) on the first generation step, and the model
    code only encodes the image when past_key_values is None. Without this patch the image is silently skipped."""
    path = f"{model_dir}/modeling_chexagent.py"
    text = open(path).read()
    if _IMAGE_BRANCH_PATCHED in text:
        return 1
    if _IMAGE_BRANCH_ORIGINAL not in text:
        raise RuntimeError("modeling_chexagent.py does not contain the expected image-branch condition")
    open(path, "w").write(text.replace(_IMAGE_BRANCH_ORIGINAL, _IMAGE_BRANCH_PATCHED))
    return 1

model_dir = snapshot_download(MODEL_NAME, local_dir="CheXagent-2-3b")
print("Patched version checks:", patch_version_asserts(model_dir))
print("Image-branch patch in place:", patch_image_cache_check(model_dir))

tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_dir, device_map="auto", torch_dtype=DTYPE, trust_remote_code=True
)
model.eval()
print("Model loaded.")


In [ ]:
GENERAL_PROMPT = "{question}"
LATERALITY_SEVERITY_PROMPT = "{question} Answer concisely based only on what is visible in the image."

def prompt_for_category(question, category):
    """Deliberately minimal (no few-shot examples, no retrieval) -- that's the
    whole point of a *baseline*. Phase 2 is where retrieved context gets added."""
    if category in ("laterality", "severity"):
        return LATERALITY_SEVERITY_PROMPT.format(question=question)
    return GENERAL_PROMPT.format(question=question)


def ask_chexagent(image_path, question, max_new_tokens=128):
    query = tokenizer.from_list_format([{"image": image_path}, {"text": question}])
    conv = [
        {"from": "system", "value": "You are a helpful radiology assistant. Answer the question as concisely as possible based only on the image."},
        {"from": "human", "value": query},
    ]
    input_ids = tokenizer.apply_chat_template(conv, add_generation_prompt=True, return_tensors="pt")
    with torch.no_grad():
        output = model.generate(
            input_ids.to(DEVICE), do_sample=False, num_beams=1, use_cache=True,
            max_new_tokens=max_new_tokens,
        )[0]
    return tokenizer.decode(output[input_ids.size(1):-1]).strip()


In [ ]:
# IMAGE CHECK: proves the picture reaches the model. The same file path is used with three different pixel patterns,
# so the text the model reads is identical every time; only the pixels differ. If the first-token scores do not change,
# the image is being ignored and nothing below this cell can be trusted.
import numpy as np
from PIL import Image

def _first_token_logprobs(path):
    query = tokenizer.from_list_format([{"image": path}, {"text": "Describe the image."}])
    conv = [
        {"from": "system", "value": "You are a helpful radiology assistant. Answer the question as concisely as possible based only on the image."},
        {"from": "human", "value": query},
    ]
    input_ids = tokenizer.apply_chat_template(conv, add_generation_prompt=True, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(input_ids.to(DEVICE), do_sample=False, num_beams=1, use_cache=True,
                             max_new_tokens=1, output_scores=True, return_dict_in_generate=True)
    return torch.log_softmax(out.scores[0][0].float(), dim=-1).cpu(), int(input_ids.size(1))

_rng = np.random.default_rng(0)
_variants = {"black": np.zeros((512, 512, 3), np.uint8), "white": np.full((512, 512, 3), 255, np.uint8),
             "noise": _rng.integers(0, 256, (512, 512, 3), dtype=np.uint8)}
_lp = {}
for _name, _pix in _variants.items():
    Image.fromarray(_pix).save("image_check.png")
    _lp[_name], _n_tokens = _first_token_logprobs("image_check.png")
_diffs = {f"{a} vs {b}": float((_lp[a] - _lp[b]).abs().max()) for a, b in (("black", "white"), ("black", "noise"), ("white", "noise"))}
print("prompt length with image slots:", _n_tokens, "tokens (context limit 2048)")
print("largest change in next-token log-probability when only the pixels change:", {k: round(v, 3) for k, v in _diffs.items()})
if max(_diffs.values()) < 0.01:
    raise RuntimeError("IMAGE CHECK FAILED: changing the pixels does not change the model's output, so the image is not "
                       "reaching the model. Stop and report this.")
print("IMAGE CHECK PASSED: the model's output depends on the pixels.")


In [ ]:
def _single_token_ids(words):
    ids = set()
    for w in words:
        try:
            t = tokenizer.encode(w, add_special_tokens=False)
        except TypeError:
            t = tokenizer.encode(w)
        if len(t) == 1:
            ids.add(t[0])
    return sorted(ids)

YES_IDS = _single_token_ids(["Yes", "yes", " Yes", " yes", "YES"])
NO_IDS = _single_token_ids(["No", "no", " No", " no", "NO"])
assert YES_IDS and NO_IDS, "could not find single-token ids for yes / no"
print("yes ids:", YES_IDS, "| no ids:", NO_IDS)


def yes_no_score(image_path, question):
    """log P(yes) - log P(no) for the first answer token (higher = more 'yes')."""
    query = tokenizer.from_list_format([{"image": image_path}, {"text": question}])
    conv = [
        {"from": "system", "value": "You are a helpful radiology assistant. Answer the question as concisely as possible based only on the image."},
        {"from": "human", "value": query},
    ]
    input_ids = tokenizer.apply_chat_template(conv, add_generation_prompt=True, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(input_ids.to(DEVICE), do_sample=False, num_beams=1, use_cache=True,
                             max_new_tokens=1, output_scores=True, return_dict_in_generate=True)
    logp = torch.log_softmax(out.scores[0][0].float(), dim=-1)
    return float(torch.logsumexp(logp[YES_IDS], 0) - torch.logsumexp(logp[NO_IDS], 0))


In [ ]:
def guess_disease(image_path, method):
    """Disease names the model gives for one image, most likely first, plus the raw text or scores.
    method: 'open' | 'forced' (free text, parsed) | 'perclass' (yes-minus-no score per candidate)."""
    if method in ("open", "forced"):
        text = ask_chexagent(image_path, DISEASE_PROMPTS[method], 48)
        return parse_diseases(text), text
    scores = {name: yes_no_score(image_path, perclass_question(name)) for name in CANDIDATES}
    return sorted(scores, key=lambda c: -scores[c]), scores


## 4. Token budget

The model reads at most 2048 tokens, and the image slots are part of that. The check measures the longest prompt including the image
slots (all arms that add retrieved text). If it exceeds the text budget the passages are shortened for **every** arm and the change is printed and
saved in `rag_settings.json`. Note it in the experiment log.


In [ ]:
TEXT_TOKEN_BUDGET = 1900   # whole prompt including the image slots; 2048 limit minus 64 new tokens and a margin

def prompt_tokens(prompt, image_path):
    tok = globals().get("tokenizer")
    if tok is None:
        return len(prompt.split()) * 2          # rough stand-in when no tokenizer is loaded
    query = tok.from_list_format([{"image": image_path}, {"text": prompt}])
    conv = [{"from": "system", "value": "You are a helpful radiology assistant. Answer the question as concisely as possible based only on the image."},
            {"from": "human", "value": query}]
    return int(tok.apply_chat_template(conv, add_generation_prompt=True, return_tensors="pt").size(1))

def longest_prompt(max_chars):
    worst = 0
    for r in know_items:
        if not r["has_oracle"]:
            continue
        ps = retrieve_context(retrievers, SOURCE, r["question"], r["knowledge_subtype"], r["oracle_disease"],
                              K, max_chars, WEIGHT)
        worst = max(worst, prompt_tokens(build_prompt(r["question"], r["oracle_disease"], ps), r["image_path"]))
    return worst

while True:
    worst = longest_prompt(MAX_CHARS)
    print(f"longest prompt at {MAX_CHARS} characters per passage: {worst} tokens (budget {TEXT_TOKEN_BUDGET})")
    if worst <= TEXT_TOKEN_BUDGET or MAX_CHARS <= 300:
        break
    MAX_CHARS -= 100
print("Using MAX_CHARS =", MAX_CHARS)


## 5. Run the arms

Disease guesses first (one per image), then every arm. Each answer is stored under a key and written to disk every 20 answers.

In [ ]:
from tqdm.auto import tqdm

CKPT = "rag_checkpoint.json"
store = json.load(open(CKPT)) if os.path.exists(CKPT) else {}
_since = 0

def cached(key, fn):
    global _since
    if key not in store:
        store[key] = fn()
        _since += 1
        if _since % 20 == 0:
            json.dump(store, open(CKPT, "w"))
    return store[key]

# --- the model's own disease guess, once per image
images = {r["image_path"]: r for r in know_items}
for path in tqdm(images, desc=f"disease guess ({DISEASE_METHOD})"):
    cached(f"guess|{DISEASE_METHOD}|{path}", lambda: guess_disease(path, DISEASE_METHOD)[1])

def disease_top(r):
    g = store[f"guess|{DISEASE_METHOD}|{r['image_path']}"]
    if DISEASE_METHOD == "perclass":
        return sorted(g, key=lambda c: -g[c])[0]
    named = parse_diseases(g)
    return named[0] if named else None

for r in know_items:
    r["guessed_disease"] = disease_top(r)
print("Guessed diseases:", dict(Counter(r["guessed_disease"] for r in know_items)))

# --- arms: (which disease, use retrieval)
ARMS = {"A1": (None, False), "A2": (None, True), "A3": ("oracle", False), "A4": ("oracle", True),
        "A5": ("guess", False), "A6": ("guess", True)}
ORACLE_ONLY = {"A2", "A3", "A4"}

def arm_input(arm, r):
    which, use_retrieval = ARMS[arm]
    disease = {None: None, "oracle": r["oracle_disease"], "guess": r["guessed_disease"]}[which]
    passages = (retrieve_context(retrievers, SOURCE, r["question"], r["knowledge_subtype"], disease, K, MAX_CHARS, WEIGHT)
                if use_retrieval else [])
    return disease, passages, build_prompt(r["question"], disease, passages)

def run_arm(arm):
    records = [r for r in know_items if r["has_oracle"]] if arm in ORACLE_ONLY else know_items
    out = []
    for r in tqdm(records, desc=arm):
        disease, passages, prompt = arm_input(arm, r)
        pred = cached(f"{arm}|{r['id']}", lambda: ask_chexagent(r["image_path"], prompt, 64))
        out.append({"arm": arm, "id": r["id"], "source_dataset": r["source_dataset"], "source_split": r["source_split"],
                    "is_dev": r["is_dev"], "knowledge_subtype": r["knowledge_subtype"], "question": r["question"],
                    "gold_answer": r["gold_answer"], "oracle_disease": r["oracle_disease"] or "",
                    "guessed_disease": r["guessed_disease"] or "", "disease_used": disease or "",
                    "passages": " || ".join(f"{p['title'][:60]} ({p['score']})" for p in passages),
                    "prediction": pred, "image_key": r["image_key"]})
    return out

all_rows = []
for arm in ARMS:
    all_rows += run_arm(arm)
json.dump(store, open(CKPT, "w"))
scored = score_knowledge_rows(all_rows)
print("Done:", len(scored), "answers")


## 6. Quick look

Means only; the formal paired comparisons, intervals and blind rating sheets are produced afterwards from the saved files
(`scripts/analyze_rag.py`). Function questions are shown separately because every gold answer is "Breathe".


In [ ]:
def mean(rows, key="item_recall"):
    return sum(float(r[key]) for r in rows) / len(rows) if rows else float("nan")

print(f"{'arm':>4} {'n (non-function)':>16} {'item recall':>12} | {'function n':>10} {'said Breathe':>12}")
for arm in ARMS:
    rows = [r for r in scored if r["arm"] == arm]
    nf = [r for r in rows if r["knowledge_subtype"] != "function"]
    fn = [r for r in rows if r["knowledge_subtype"] == "function"]
    breathe = sum("breathe" in r["prediction"].lower() for r in fn)
    print(f"{arm:>4} {len(nf):>16} {mean(nf):>12.2f} | {len(fn):>10} {breathe:>12}")

print("\nOracle subset only (same questions in every arm):")
oracle_ids = {r["id"] for r in know_items if r["has_oracle"]}
for arm in ARMS:
    rows = [r for r in scored if r["arm"] == arm and r["id"] in oracle_ids]
    print(f"{arm:>4} n={len(rows):>3} item recall {mean(rows):.2f}")

known = [r for r in know_items if r["has_oracle"]]
hit = sum(r["guessed_disease"] == r["oracle_disease"] for r in known)
print(f"\nThe model's disease guess matches the true disease on {hit} / {len(known)} oracle-subset questions")


## 7. Save results

Keep these files. The Colab VM is wiped when the session ends.

In [ ]:
import csv

fields = ["arm", "id", "source_dataset", "source_split", "is_dev", "knowledge_subtype", "question", "gold_answer",
          "oracle_disease", "guessed_disease", "disease_used", "passages", "prediction", "image_key",
          "item_recall", "item_precision", "item_f1", "token_f1"]
with open("rag_results.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields)
    w.writeheader()
    for row in scored:
        w.writerow({k: row.get(k, "") for k in fields})
json.dump({"disease_method": DISEASE_METHOD, "source": SOURCE, "k": K, "max_chars": MAX_CHARS, "weight": WEIGHT,
           "scorer_version": SCORER_VERSION, "quick": QUICK}, open("rag_settings.json", "w"), indent=1)
print(f"Saved rag_results.csv ({len(scored)} rows) and rag_settings.json")
try:
    from google.colab import files
    for name in ("rag_results.csv", "rag_settings.json", "rag_checkpoint.json"):
        files.download(name)
except Exception as exc:
    print("Auto-download unavailable; use the Colab file browser instead.", exc)
